# Objective-Driven Agentic Nanoindentation — Main Orchestrator + Bayesian Optimization

This notebook upgrades the visual multi-agent workflow (notebook 3) from a **fixed pipeline** to an
**objective-driven architecture**: you state a high-level research goal in plain English, and a
**Main Orchestrator Agent** compiles it into a machine-actionable objective, briefs every sub-agent,
selects indentation sites with **Gaussian-process Bayesian optimization**, evaluates progress after
every round, and adapts the strategy — *the only input that changes between experiments is your goal.*

```
                    ┌─────────────────────────────────────────────┐
 USER OBJECTIVE ──► │ 🎯 MAIN ORCHESTRATOR                        │ ◄── feedback loop
 ("find regions     │  compile objective → 🛑 GATE 0 (you)        │      (progress, EI,
  with higher H     │  brief all sub-agents                       │       budget, critic)
  and higher E")    └───────┬─────────────────────────────────────┘
                            │ per-role directives
   ┌────────────┬───────────┼─────────────┬──────────────┐
   ▼            ▼           ▼             ▼              ▼
 🔭 Observe  🔍 Image    📚 Literature  💡 Hypothesis  📊 Measurement
             Analysis                   ⇄ ⚖ Critic     Critic (objective
             (objective-               🛑 GATE 1        criteria)
              ranked targets)
                            │
        round 1: top objective-ranked targets → 🛑 GATE 2 → 🦾 execute
                            │
        ┌───────────────────▼────────────────────────────────────┐
        │  ORCHESTRATOR ROUND LOOP (re-runnable cells A→D)       │
        │  evaluate progress → decide → 📈 GP + Expected          │
        │  Improvement proposes sites → 🛑 GATE → execute →       │
        │  critic re-evaluates → repeat / stop                    │
        └────────────────────────────────────────────────────────┘
```

**Safety model is unchanged from notebook 3:** every instrument action still waits for an explicit
human approval gate, plus a new **Gate 0** where you approve the orchestrator's *interpretation of
your goal* before anything else happens. Runs offline against the built-in microstructure simulator
by default; `USE_SIMULATOR = False` drives the real iMicro with the identical code path.

## 1. Setup — your research objective, switches, credentials

**`USER_OBJECTIVE` is the only line you need to change to run a different experiment.**
The orchestrator parses it (LLM when available, deterministic parser otherwise), you approve the
interpretation at Gate 0, and every downstream agent adapts automatically.

`AUTO_APPROVE_ALL = False` (default) keeps every gate manual. `True` runs unattended
(simulator only — a guard blocks full-auto on real hardware). Environment overrides used by
headless runs: `NB5_OBJECTIVE`, `NB5_AUTO_APPROVE=1`, `NB5_DISABLE_LLM=1`.

In [ ]:
import os, sys, json, time, warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
pd.set_option("display.max_colwidth", 90)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "nanoindentation_agents").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# ------------------------- the research objective -------------------------
# THE ONLY REQUIRED INPUT. State the goal; the orchestrator does the rest.
USER_OBJECTIVE = os.environ.get("NB5_OBJECTIVE",
    "Find the regions with the highest hardness AND highest elastic modulus on this "
    "compositionally-varying steel combinatorial library of unknown composition, "
    "using as few indents as possible."

)

# ------------------------- operator switches -------------------------
OPERATOR_NAME        = "operator"   # recorded on approval decisions
USE_SIMULATOR        = False                    # False -> real iMicro Flask server
RUN_SIMULATOR_SERVER = False                    # start the simulator from this notebook
AUTO_APPROVE_ALL     = os.environ.get("NB5_AUTO_APPROVE", "0") == "1"   # unattended run (SIMULATOR ONLY)
AUTO_RUN_ROUNDS      = os.environ.get("NB5_AUTO_RUN_ROUNDS", "0") == "1"  # gates 0-2 stay manual, but the
                                               # Section 14 loop runs ALL remaining rounds hands-free (SIMULATOR ONLY)
RUN_AUTONOMOUS_CAMPAIGN = os.environ.get("NB5_AUTONOMOUS", "0") == "1"    # Section 14 FULLY HANDS-FREE loop on
                                               # REAL hardware: auto-approves EVERY irreversible round and runs
                                               # until INDENT_BUDGET / MAX_ROUNDS / objective_met. Leave False
                                               # to approve each round manually (cells A-D).
FOV_W, FOV_H         = 100.0, 75.0             # video-panel field of view in µm
RUN_ID               = os.environ.get("NB5_RUN_ID") or time.strftime("%m%d_%H%M%S")  # unique per run -> keeps iMicro ReviewData window titles from colliding

INDENT_BUDGET = int(os.environ.get("NB5_INDENT_BUDGET", "16"))   # total indents the orchestrator may spend
MAX_ROUNDS    = int(os.environ.get("NB5_MAX_ROUNDS", "5"))       # adaptive rounds after round 1
os.environ["NB5_TEST_TIMEOUT_S"] = "1800"  # 30 min per indent — real hardware runs longer than the 210s default

# Safety guard: unattended (auto-approved) runs are never allowed on real hardware.
assert USE_SIMULATOR or not AUTO_APPROVE_ALL, (
    "Full-auto is only allowed in simulator mode — a human must approve real indents."
)
assert USE_SIMULATOR or not AUTO_RUN_ROUNDS, (
    "Hands-free adaptive rounds are only allowed in simulator mode."
)

# RUN_AUTONOMOUS_CAMPAIGN is intentionally permitted on real hardware (operator opt-in);
# it is the only switch that auto-approves irreversible indents without a per-round gate.
if RUN_AUTONOMOUS_CAMPAIGN and not USE_SIMULATOR:
    print("\u26a0\ufe0f  AUTONOMOUS real-hardware campaign ENABLED — Section 14 will auto-approve and\n"
          "    run EVERY adaptive round (irreversible indents) with NO per-round human gate,\n"
          "    stopping only at INDENT_BUDGET / MAX_ROUNDS / objective_met.")

from nanoindentation_agents.notebook_credentials import (
    load_project_env, resolve_hf_token, resolve_instrument_token,
)
load_project_env(REPO_ROOT)

INSTRUMENT_HOST  = os.environ.get("NANOINDENT_HOST", "YOUR_INSTRUMENT_IP") #simulator"127.0.0.1" and Imicro"YOUR_INSTRUMENT_IP"
# Simulator defaults to 5005 (macOS AirPlay occupies 5000); the real iMicro server uses 5000.
INSTRUMENT_PORT  = int(os.environ.get("NANOINDENT_PORT", "5005" if USE_SIMULATOR else "5000"))
INSTRUMENT_TOKEN = resolve_instrument_token()
HF_TOKEN_OK      = bool(resolve_hf_token())
DISABLE_LLM      = os.environ.get("NB5_DISABLE_LLM", "0") == "1"
TRY_LLM          = HF_TOKEN_OK and not DISABLE_LLM      # orchestrator/hypothesis/literature LLM calls
TRY_VISION_LLM   = TRY_LLM                                # vision path shares the switch
# Need to adjust the Nanoindent_token in the notebook_credientatial to connect to the iMicro server. The default is "xxxxxxxxxxx" 
from nanoindentation_agents import notebook_ui as ui
from nanoindentation_agents.state import create_initial_state
from nanoindentation_agents.config import load_config, set_active_config
from nanoindentation_agents.tools.indent_tools import IndentToolkit, set_toolkit
from nanoindentation_agents.agents.observe_agent import observe_node
from nanoindentation_agents.agents.literature_agent import literature_node
from nanoindentation_agents.agents.hypothesis_agent import hypothesis_node
from nanoindentation_agents.critics.hypothesis_critic import hypothesis_critic_node
from nanoindentation_agents.critics.measurement_critic import measurement_critic_node, get_routing_decision
from nanoindentation_agents.analysis.spatial_targeting import (
    sync_targets_from_state, pick_next_target, target_visit_key,
)
from nanoindentation_agents.analysis.exploration_plan import refresh_exploration_plan

# --- the objective-driven layer (new in this notebook) ---
from nanoindentation_agents.objective import compile_objective
from nanoindentation_agents.agents.orchestrator_agent import OrchestratorAgent
from nanoindentation_agents.analysis.objective_acquisition import (
    HAVE_SKLEARN, ObjectiveSurrogate, rank_targets_by_objective,
)

print(f"Repo:            {REPO_ROOT}")
print(f"Mode:            {'SIMULATOR' if USE_SIMULATOR else 'REAL iMicro'}")
print(f"Instrument:      http://{INSTRUMENT_HOST}:{INSTRUMENT_PORT}")
print(f"HF token loaded: {HF_TOKEN_OK}  (LLM calls {'enabled' if TRY_LLM else 'disabled -> deterministic fallbacks'})")
print(f"sklearn GP:      {'available' if HAVE_SKLEARN else 'MISSING -> inverse-distance heuristic'}")
print(f"Operator:        {OPERATOR_NAME}")
print(f"Approval mode:   {'AUTO — all gates auto-approve (unattended)' if AUTO_APPROVE_ALL else 'MANUAL — operator approves every gate'}")
print(f"Round loop:      {'AUTO — Section 14 runs remaining rounds hands-free' if (AUTO_APPROVE_ALL or AUTO_RUN_ROUNDS) else 'MANUAL — re-run cells A-D once per round'}")
print(f"Budget:          {INDENT_BUDGET} indents, {MAX_ROUNDS} adaptive rounds max")
print()
print("OBJECTIVE:", USER_OBJECTIVE)

In [ ]:
# Start (or reuse) the instrument server and verify the connection.
import subprocess

SIM_DATA_DIR = REPO_ROOT / "sim_instrument_data"
DOWNLOAD_DIR = REPO_ROOT / "notebooks" / "downloaded_data"
ANNOTATED_DIR = REPO_ROOT / "notebooks" / "annotated_images"

sim_process = globals().get("sim_process")
if USE_SIMULATOR and RUN_SIMULATOR_SERVER:
    if sim_process is None or sim_process.poll() is not None:
        sim_process = subprocess.Popen(
            [sys.executable, str(REPO_ROOT / "scripts" / "run_sim_server.py"),
             "--host", INSTRUMENT_HOST, "--port", str(INSTRUMENT_PORT),
             "--data-dir", str(SIM_DATA_DIR)],
            cwd=str(REPO_ROOT), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT,
        )
        time.sleep(1.5)
        print(f"Simulator server started (PID {sim_process.pid}) — data: {SIM_DATA_DIR}")
    else:
        print(f"Simulator already running (PID {sim_process.pid})")

cfg = load_config(None)
cfg.research_question = USER_OBJECTIVE
cfg.experiment_context = USER_OBJECTIVE
cfg.instrument.host = INSTRUMENT_HOST
cfg.instrument.port = INSTRUMENT_PORT
cfg.instrument.token = INSTRUMENT_TOKEN
cfg.instrument.sim_mode = USE_SIMULATOR
cfg.literature.local_dir = str(REPO_ROOT / "literature")
cfg.memory.base_dir = str(REPO_ROOT / "experiment_memory")
set_active_config(cfg)

toolkit = IndentToolkit(host=INSTRUMENT_HOST, port=INSTRUMENT_PORT,
                        token=INSTRUMENT_TOKEN, timeout=cfg.instrument.timeout)
set_toolkit(toolkit)

assert toolkit.ping(), "Instrument server unreachable — check section 1 switches / server."
print("Instrument server connection: OK")

## 2. The agent team — now led by a Main Orchestrator

| Agent | Responsibility | Talks to |
|---|---|---|
| 🎯 **Main Orchestrator** | **Compiles your goal into a structured objective; briefs every agent; evaluates progress each round; decides continue / refine / stop** | everyone |
| 📈 **Acquisition Agent (BO)** | GP surrogate of hardness & modulus over (x, y); proposes next sites by Expected Improvement | Orchestrator, Planning |
| 🔭 Observe Agent | Captures the iMicro video panel + instrument state | Image Analysis |
| 🔍 Image Analysis Agent | Segments phases/boundaries/pores; targets **re-ranked by the objective** | Orchestrator, Planning |
| 📚 Literature Agent | Search query **pre-seeded by the orchestrator** from the objective | Hypothesis |
| 💡 Hypothesis Agent | Testable hypothesis **directed at the objective** | Hypothesis Critic |
| ⚖ Hypothesis Critic | Reviews testability and specificity | Hypothesis, Operator |
| 📋 Planning Agent | Converts targets into instrument action plans (grid/budget from objective constraints) | Operator (gate) |
| 🧑 **Human Operator** | **Approves the objective (Gate 0) and every instrument action** | everyone |
| 🦾 Instrument Executor | Runs approved tool calls through the Flask server | Measurement Critic |
| 📊 Measurement Critic | Judges evidence against **objective-derived success criteria** | Orchestrator |

The deterministic supervisor heuristics of notebook 3 are replaced by the orchestrator's
**bounded decision loop**: it can only choose among `continue`, `refine_hypothesis`,
`objective_met`, `budget_exhausted`, `escalate_to_human` — rules are authoritative and an LLM
may only advise on the soft choices. Every decision is logged with its provenance.

In [ ]:
# Fresh experiment state + agent-to-agent message log.
state = create_initial_state(
    research_question=USER_OBJECTIVE,
    experiment_name="objective_driven_orchestrator",
    max_iterations=cfg.experiment.max_iterations,
    ic_max_cycles=cfg.experiment.ic_max_cycles,
)
msg_log = ui.AgentMessageLog()
executions = []          # raw per-array execution records
ui.ApprovalGate.registry.clear()

roles = ["orchestrator", "image_analysis", "literature", "hypothesis",
         "hypothesis_critic", "measurement_critic"]
roster = []
for r in roles:
    try:
        spec = cfg.models.role(r)
        roster.append({"agent role": r, "LLM backing": spec.model, "vision": spec.vision,
                       "provider": spec.provider, "fallback": "deterministic rule/CV path"})
    except Exception:
        roster.append({"agent role": r, "LLM backing": "-", "vision": False,
                       "provider": "-", "fallback": "deterministic rule/CV path"})
display(pd.DataFrame(roster))
print("State initialized — the orchestrator seeds it with the objective in the next section.")

## 3. 🎯 Main Orchestrator — compile the objective

The orchestrator translates your free-text goal into a **structured `ResearchObjective`**:
which properties to optimize and in which direction, their relative weights, the indent budget,
and quantitative success criteria. With an HF token it uses the `orchestrator` LLM role
(strict-JSON prompt, validated and clamped field by field); without one, a deterministic keyword
parser produces the same structure. Either way **you approve the interpretation at Gate 0** before
anything else runs.

In [ ]:
ui.agent_banner("Orchestrator Agent", "compiling the research objective into a machine-actionable target")

objective = compile_objective(
    USER_OBJECTIVE, cfg=cfg,
    fov_bounds=(0.0, FOV_W, 0.0, FOV_H),      # refined to stage coordinates after the first capture
    indent_budget=INDENT_BUDGET, max_rounds=MAX_ROUNDS,
    use_llm=TRY_LLM,
)

# --- TESTING: force a single 1x1 indent (overrides the 3x3 ObjectiveConstraints default) ---
objective.constraints.grid_nx = 2
objective.constraints.grid_ny = 2
objective.constraints.spacing_um = 15 
orch = OrchestratorAgent(objective, cfg, use_llm=TRY_LLM)
state.update(orch.objective_state_updates())

goals_df = pd.DataFrame([{
    "property": g.name, "direction": g.direction, "weight": g.weight,
    "target range (GPa)": str(g.target_range) if g.target_range else "-",
} for g in objective.property_goals])
display(goals_df)
print(f"Material system:  {objective.material_system or '(not stated)'}")
print(f"Compiled by:      {objective.compiled_by.upper()}")
print(f"Budget:           {objective.constraints.indent_budget} indents | "
      f"{objective.constraints.max_rounds} adaptive rounds | "
      f"{objective.constraints.grid_nx}x{objective.constraints.grid_ny} arrays "
      f"@ {objective.constraints.spacing_um:g} µm")
print(f"Success criteria: best site >= {objective.success.min_improvement_pct:g}% above baseline, "
      f">= {objective.success.min_confirming_arrays} arrays, "
      f">= {objective.success.min_visited_locations} locations")
print("Stop conditions:  " + "; ".join(objective.stop_conditions))

msg_log.log("Human Operator", "Orchestrator Agent",
            f"Research objective submitted: {USER_OBJECTIVE[:120]}",
            detail=USER_OBJECTIVE)
msg_log.log("Orchestrator Agent", "Human Operator",
            f"Objective compiled ({objective.compiled_by}) — "
            f"{'; '.join(g.describe() for g in objective.property_goals)}",
            detail=objective.summary_text()[:300])

## 4. 🛑 Gate 0 — approve the machine's understanding of your goal

New in the objective-driven design: before any observation, planning or indentation, **you confirm
that the orchestrator understood the objective correctly**. If the parse looks wrong (wrong
direction, wrong property, wrong budget), reject the gate, refine `USER_OBJECTIVE` in section 1
and re-run from section 3.

In [ ]:
objective_gate = ui.ApprovalGate(
    "gate-0-objective", "Research objective approval",
    description=("Approve the orchestrator's interpretation of your research goal. "
                 "All sub-agent briefings, success criteria and the indent budget derive "
                 "from this objective.\n\n" + objective.summary_text()),
)

In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE <==
# Type your decision, then run this cell:
#     MY_DECISION = "approve"                    (optionally: "approve: your note")
#     MY_DECISION = "reject: <your reason>"
MY_DECISION = "approve"

if AUTO_APPROVE_ALL:
    objective_gate.approve(approved_by="AUTO (unattended run)",
        note="Auto-approved: AUTO_APPROVE_ALL is enabled in section 1")
elif MY_DECISION.strip().lower().startswith("approve"):
    objective_gate.approve(approved_by=OPERATOR_NAME,
        note=MY_DECISION.partition(":")[2].strip() or "Approved by operator")
elif MY_DECISION.strip().lower().startswith("reject"):
    objective_gate.reject(reason=MY_DECISION.partition(":")[2].strip() or "Rejected by operator",
        approved_by=OPERATOR_NAME)
else:
    print("PAUSED — this gate is still PENDING and the workflow will not continue.")
    print('  Edit this cell: set MY_DECISION = "approve" or "reject: <reason>", then re-run it.')

if objective_gate.decision:
    _approver = "AUTO Approver" if AUTO_APPROVE_ALL else "Human Operator"
    msg_log.log(_approver, "Orchestrator Agent",
                "Objective APPROVED — brief the sub-agents" if objective_gate.is_approved
                else "Objective REJECTED — refine USER_OBJECTIVE and re-run section 3",
                detail=objective_gate.note)

In [ ]:
objective_gate.require_approved()
ui.agent_banner("Orchestrator Agent", "briefing every sub-agent on the approved objective")

state.update(orch.briefing_state_updates(state))

briefings = state["orchestrator_briefings"]
display(pd.DataFrame([{"agent role": role, "orchestrator directive": text}
                      for role, text in briefings.items()]))
print(f"Literature search pre-seeded: {state['literature_search_query']!r}")

_role_names = {
    "image_analysis": "Image Analysis Agent", "literature": "Literature Agent",
    "hypothesis": "Hypothesis Agent", "planning": "Planning Agent",
    "measurement_critic": "Measurement Critic",
}
for role, text in briefings.items():
    msg_log.log("Orchestrator Agent", _role_names.get(role, role),
                f"Objective briefing: {text[:110]}", detail=text)

## 5. 🔭 Observe Agent — capture the sample

Identical to notebook 3; additionally, the first capture fixes the **stage-coordinate bounds of
the search area** the acquisition agent may propose sites in.

In [ ]:
ui.agent_banner("Observe Agent", "reading instrument state and capturing the initial iMicro frame")

state.update(observe_node(state))

# --- Scale calibration from the burned-in scale bar -------------------------
# The real video-panel FOV depends on the current objective/zoom, so derive it
# from the red scale bar each run instead of trusting a hardcoded FOV. This sets
# the um-per-pixel used to convert image targets into stage moves. HARD-STOP on
# a bad calibration -- never move the indenter on an unknown scale. (The
# simulator's synthetic frame has no scale bar, so keep the nominal FOV there.)
if not USE_SIMULATOR:
    _cal_img = ui.resolve_image_path(state.get("current_image") or {})
    assert _cal_img, "No captured frame to calibrate against -- re-run the Observe stage."
    _um_per_px, _cal = ui.calibrate_um_per_px(_cal_img)   # raises if the bar can't be read
    FOV_W, FOV_H = _cal["fov_w_um"], _cal["fov_h_um"]
    print(f"Scale calibration: bar {_cal['bar_len_px']}px = {_cal['bar_um']} um "
          f"({_cal['bar_um_source']}) -> {_um_per_px:.4f} um/px  |  "
          f"FOV {FOV_W:.1f} x {FOV_H:.1f} um  (image {_cal['img_w']}x{_cal['img_h']}px)")
else:
    _um_per_px = None
    print(f"Simulator mode -- using nominal FOV {FOV_W:.1f} x {FOV_H:.1f} um (no scale bar to calibrate).")

if state.get("current_image"):
    state["current_image"]["width_um"] = FOV_W
    state["current_image"]["height_um"] = FOV_H
    if _um_per_px is not None:
        state["current_image"]["um_per_px"] = _um_per_px

pos = state.get("current_position", (0.0, 0.0))
capture_pos = tuple(pos)                      # FOV center -- fixed reference for all rounds

# The BO search area = this field of view, in stage coordinates.
objective.constraints.fov_bounds = (
    capture_pos[0] - FOV_W / 2, capture_pos[0] + FOV_W / 2,
    capture_pos[1] - FOV_H / 2, capture_pos[1] + FOV_H / 2,
)
state["research_objective"] = objective.to_dict()

msg_log.log("Observe Agent", "Image Analysis Agent",
            f"Captured video panel at stage ({pos[0]:.1f}, {pos[1]:.1f}) um",
            detail=f"local file: {state['current_image'].get('local_path') or state['current_image'].get('image_path')}")

ui.show_capture(state["current_image"],
                title="Initial iMicro video panel (as-captured)",
                note="This frame defines the orchestrator's search area for the whole campaign.",
                fov_width_um=FOV_W, fov_height_um=FOV_H)
print(f"BO search bounds (stage um): x {objective.constraints.fov_bounds[0]:.0f}..{objective.constraints.fov_bounds[1]:.0f}, "
      f"y {objective.constraints.fov_bounds[2]:.0f}..{objective.constraints.fov_bounds[3]:.0f}")

## 6. 🔍 Image Analysis — briefed by the orchestrator, ranked by the objective

Same two cross-checked paths as notebook 3 (vision LLM + classical CV), but with two differences:

1. The vision prompt carries the **orchestrator's directive** for this role.
2. The fixed priority queue is replaced by **`rank_targets_by_objective`** — targets are scored by
   how well their expected properties align with the goal, so a *maximize-hardness* objective puts
   the bright second phase first while a *find-softest-regions* objective would rank the matrix
   first, with **zero code changes**.

In [ ]:
ui.agent_banner("Image Analysis Agent", "segmenting microstructure — directive: " +
                briefings["image_analysis"][:80])

img_path = ui.resolve_image_path(state["current_image"])
assert img_path, "No captured image found — re-run the Observe stage."

cv_result = ui.classical_cv_analysis(img_path, FOV_W, FOV_H)

vision_result = None
if TRY_VISION_LLM:
    from nanoindentation_agents.image_analysis.vision_agent import analyze_image_with_vision
    vision_result = ui.run_with_timeout(
        lambda: analyze_image_with_vision(
            str(img_path), research_question=state["research_question"],
            experiment_context=state["experiment_context"]
                + "\n\n[ORCHESTRATOR DIRECTIVE]\n" + briefings["image_analysis"],
            cfg=cfg),
        seconds=120, label="Vision LLM (Qwen2.5-VL)")
    if vision_result and not vision_result.get("vision", {}).get("ok"):
        print("Vision LLM unavailable:", vision_result["vision"].get("error", "")[:120])

analysis, provenance = ui.combine_image_analysis(cv_result, vision_result)
state["image_analysis"] = analysis
state.update(sync_targets_from_state(state))
state.update(refresh_exploration_plan(state))

# --- objective-aware ranking replaces the fixed priority queue ---
ranked = rank_targets_by_objective(state["measurement_plan"], objective, analysis)
state["measurement_plan"] = ranked

print(f"Analysis provenance: {provenance}\n")
print("Agent interpretation:")
print(analysis["interpretation"])

display(pd.DataFrame([{
    "rank": t["priority"], "target": t.get("label", "?"),
    "x (µm)": round(t.get("x_um", 0), 1), "y (µm)": round(t.get("y_um", 0), 1),
    "objective score": t["objective_score"], "why": t["rank_reason"],
} for t in ranked[:6]]))

msg_log.log("Image Analysis Agent", "Orchestrator Agent",
            f"{len(ranked)} targets ranked by objective — top: "
            f"{', '.join(t.get('label', '?') for t in ranked[:3])}",
            detail=analysis["interpretation"][:300])

ui.annotate_targets(state["current_image"], analysis, ranked[:4],
                    fov_width_um=FOV_W, fov_height_um=FOV_H,
                    save_path=ANNOTATED_DIR / "nb5_01_objective_ranked_targets.png")

## 7. 📚 Literature Agent — query pre-seeded by the orchestrator

The orchestrator already wrote `literature_search_query` / `literature_search_keywords` into state
from the objective and material system — the literature agent detects the pre-seeded query and
searches exactly what the objective needs (this is the zero-code-change injection point).

In [ ]:
ui.agent_banner("Literature Agent", f"searching: {state['literature_search_query']!r}")

lit_updates = ui.run_with_timeout(lambda: literature_node(state), seconds=240, label="Literature Agent")
if not lit_updates or not (lit_updates.get("literature_context") or "").strip():
    # Fully offline fallback: retrieve from the local ./literature notes directly.
    from nanoindentation_agents.rag.literature_retriever import LiteratureRetriever
    ctx, srcs = LiteratureRetriever(cfg.literature).retrieve(
        state["literature_search_query"],
        keywords=state.get("literature_search_keywords") or ["hardness", "modulus"])
    lit_updates = {
        "literature_context": ctx[:2500], "literature_raw_context": ctx,
        "literature_sources": srcs,
        "literature_search_meta": {"mode": "local_files (offline fallback)"},
    }
state.update(lit_updates)

# Quantitative expectation bands for the final comparison plots (same sample class as notebook 3).
LIT_EXPECTATIONS = {                      # hardness bands, GPa
    "hard_phase_center": (9.0, 14.0),
    "hard_phase_2":      (9.0, 14.0),
    "matrix_reference":  (5.0, 8.0),
    "grain_boundary":    (6.0, 10.0),
}

ui.literature_panel(state)
msg_log.log("Literature Agent", "Hypothesis Agent",
            f"Context from {len(state.get('literature_sources', []))} source(s) for the "
            f"objective-directed query",
            detail=(state.get("literature_context") or "")[:280])

## 8. 💡 Hypothesis Agent ⇄ ⚖ Hypothesis Critic — objective-directed

The hypothesis prompt now contains the orchestrator's directive (via `orchestrator_briefings` in
state), so the hypothesis is about *which regions satisfy the objective*, with the success margin
taken from the objective's criteria rather than a hardcoded 10%.

In [ ]:
ui.agent_banner("Hypothesis Agent", "drafting an objective-directed hypothesis")

hyp_updates = ui.run_with_timeout(lambda: hypothesis_node(state), seconds=240, label="Hypothesis Agent")
if not hyp_updates or not (hyp_updates.get("hypothesis") or "").strip():
    _goal = "; ".join(g.describe() for g in objective.property_goals)
    _pct = objective.success.min_improvement_pct
    hyp = (f"We hypothesize that specific microstructural regions of the "
           f"{objective.material_system or 'sample'} best satisfy the objective ({_goal}): "
           f"the bright second-phase grains are expected to maximize hardness (9-14 GPa) and "
           f"modulus (~200-220 GPa) relative to the darker matrix (5-8 GPa, ~160-175 GPa). "
           f"The objective is met if the best measured site beats the sample baseline by "
           f">= {_pct:g}% on the weighted property score; uniform properties refute it.")
    hyp_updates = {
        "hypothesis": hyp, "hypothesis_history": state["hypothesis_history"] + [hyp],
        "hypothesis_data": {
            "hypothesis": hyp,
            "rationale": "Image analysis shows two-phase contrast; objective weighting favors the high-H/high-E phase.",
            "measurements": (f"{objective.constraints.grid_nx}x{objective.constraints.grid_ny} arrays at the "
                             f"top objective-ranked targets, then GP/EI-selected sites."),
            "success_criteria": f"Best site >= {_pct:g}% above baseline on the weighted objective score.",
            "failure_criteria": "No site beats the baseline by the required margin within budget.",
        },
    }
state.update(hyp_updates)
ui.hypothesis_panel(state)

# --- critic debate loop ---
for _ in range(3):
    state.update(hypothesis_critic_node(state))
    approved = state["hypothesis_critique_approved"]
    ui.critic_panel(f"Hypothesis Critic — pass {state['hypothesis_critic_passes']}",
                    state["hypothesis_critique"], approved)
    if approved and state["hypothesis_critic_passes"] >= cfg.experiment.min_hypothesis_critic_passes:
        msg_log.log("Hypothesis Critic", "Human Operator",
                    "Hypothesis approved by critic — forwarding for operator decision",
                    detail=state["hypothesis_critique"][:200])
        break
    msg_log.log("Hypothesis Critic", "Hypothesis Agent",
                "Revision requested" if not approved else "Second critic pass required",
                detail=state["hypothesis_critique"][:200])
    upd = ui.run_with_timeout(lambda: hypothesis_node(state), seconds=240, label="Hypothesis Agent (revision)")
    if upd:
        state.update(upd)
        ui.hypothesis_panel(state)

## 9. 🛑 Gate 1 — hypothesis sign-off

Same contract as notebook 3: approve to continue to planning, reject to block downstream cells.

In [ ]:
hypothesis_gate = ui.ApprovalGate(
    "gate-1-hypothesis", "Hypothesis approval",
    description=("Approve the objective-directed hypothesis before any instrument planning begins. "
                 "No stage motion or indentation is possible until later gates are also approved."),
)

In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE <==
# Type your decision, then run this cell:
#     MY_DECISION = "approve"                    (optionally: "approve: your note")
#     MY_DECISION = "reject: <your reason>"
MY_DECISION = "approve"

if AUTO_APPROVE_ALL:
    hypothesis_gate.approve(approved_by="AUTO (unattended run)",
        note="Auto-approved: AUTO_APPROVE_ALL is enabled in section 1")
elif MY_DECISION.strip().lower().startswith("approve"):
    hypothesis_gate.approve(approved_by=OPERATOR_NAME,
        note=MY_DECISION.partition(":")[2].strip() or "Approved by operator")
elif MY_DECISION.strip().lower().startswith("reject"):
    hypothesis_gate.reject(reason=MY_DECISION.partition(":")[2].strip() or "Rejected by operator",
        approved_by=OPERATOR_NAME)
else:
    print("PAUSED — this gate is still PENDING and the workflow will not continue.")
    print('  Edit this cell: set MY_DECISION = "approve" or "reject: <reason>", then re-run it.')

if hypothesis_gate.decision:
    state["user_approved"] = hypothesis_gate.is_approved
    _approver = "AUTO Approver" if AUTO_APPROVE_ALL else "Human Operator"
    msg_log.log(_approver, "Planning Agent",
                "Hypothesis APPROVED — proceed to experiment planning" if hypothesis_gate.is_approved
                else "Decision: REJECTED — downstream actions blocked",
                detail=hypothesis_gate.note)

## 10. 📋 Round 1 — plan the top objective-ranked targets

The Planning Agent takes the **two best targets from the objective ranking** (not a fixed
hard-phase/matrix pair) and applies the **grid geometry and budget from the objective's
constraints**. The orchestrator's budget preview shows what this round costs.

In [ ]:
hypothesis_gate.require_approved()
ui.agent_banner("Planning Agent", "planning arrays at the top objective-ranked targets")

c = objective.constraints
round1_targets = [dict(t) for t in state["measurement_plan"][:1]]
round1_plans = []
for t in round1_targets:
    plan = ui.plan_target_actions(
        t, capture_pos, sample_name=f"r1_{t['label']}_{RUN_ID}",
        grid_nx=c.grid_nx, grid_ny=c.grid_ny, spacing_um=c.spacing_um,
        setup=dict(cfg.indent_defaults.setup), method_type=cfg.indent_defaults.method_type,
        fov_width_um=FOV_W, fov_height_um=FOV_H,
        expected_outcome=f"Objective rank {t['priority']}: {t['rank_reason']}",
    )
    round1_plans.append(plan)
    print(f"Plan '{plan['sample_name']}': move to stage {plan['destination']} µm, "
          f"{plan['grid_nx']}x{plan['grid_ny']} array @ {plan['spacing_um']} µm — {t['rank_reason']}")

_planned = sum(p["grid_nx"] * p["grid_ny"] for p in round1_plans)
print(f"\nBudget preview: {_planned} indents this round / {c.indent_budget} total budget")

ui.plot_planned_grids(state["current_image"], round1_plans, capture_pos,
                      title="Round 1 — arrays at the top objective-ranked targets",
                      fov_width_um=FOV_W, fov_height_um=FOV_H)
msg_log.log("Planning Agent", "Human Operator",
            f"Requesting approval: {len(round1_plans)} indent arrays ({_planned} indents) at the "
            f"top objective-ranked targets",
            detail="; ".join(p["justification"][:120] for p in round1_plans))

## 11. 🛑 Gate 2 — indentation is irreversible

Nothing executes until you approve. To veto a single region, reject, remove that plan from
`round1_plans`, and re-run from section 10.

In [ ]:
indent_gate_1 = ui.ApprovalGate(
    "gate-2-round1-indents", "Round 1 — arrays at the top objective-ranked targets",
    description=(f"Approving executes, per region: stage move -> confirmation capture -> "
                 f"{objective.constraints.grid_nx}x{objective.constraints.grid_ny} indent array -> "
                 f"monitoring -> ReviewData export. "
                 f"Total: {sum(p['grid_nx']*p['grid_ny'] for p in round1_plans)} indents "
                 f"at {len(round1_plans)} stage locations."),
    action_cards=[card for p in round1_plans for card in p["cards"]],
)

In [ ]:
# ==> YOUR DECISION IS REQUIRED HERE <==
# Type your decision, then run this cell:
#     MY_DECISION = "approve"                    (optionally: "approve: your note")
#     MY_DECISION = "reject: <your reason>"
MY_DECISION = "approve"

if AUTO_APPROVE_ALL:
    indent_gate_1.approve(approved_by="AUTO (unattended run)",
        note="Auto-approved: AUTO_APPROVE_ALL is enabled in section 1")
elif MY_DECISION.strip().lower().startswith("approve"):
    indent_gate_1.approve(approved_by=OPERATOR_NAME,
        note=MY_DECISION.partition(":")[2].strip() or "Approved by operator")
elif MY_DECISION.strip().lower().startswith("reject"):
    indent_gate_1.reject(reason=MY_DECISION.partition(":")[2].strip() or "Rejected by operator",
        approved_by=OPERATOR_NAME)
else:
    print("PAUSED — this gate is still PENDING and the workflow will not continue.")
    print('  Edit this cell: set MY_DECISION = "approve" or "reject: <reason>", then re-run it.')

if indent_gate_1.decision:
    _approver = "AUTO Approver" if AUTO_APPROVE_ALL else "Human Operator"
    msg_log.log(_approver, "Instrument Executor",
                "Round-1 instrument actions APPROVED — execute exactly as carded" if indent_gate_1.is_approved
                else "Decision: REJECTED — downstream actions blocked",
                detail=indent_gate_1.note)

## 12. 🦾 Instrument Executor — run round 1

### 12.1  Screen-reading fallback for live iMicro results (REAL hardware)

The **primary** source of results stays the exported **result-file download**
(`ui.execute_target_plan` → `process_review_file` → CSV). This section adds a
**safety net** that is used **only when the notebook cannot retrieve valid
results from the iMicro computer** (export missing, empty, or all-`NaN`).

While the array runs, the notebook continuously reads the live left-hand numeric
panel — **DEPTH / HARDNESS / MODULUS** — which the iMicro Flask server already
OCR/UI-reads and returns through `get_all_ui_state`. It then repackages those
screen readings into the **identical `indent_data` shape** the agent already
consumes, so the round-2 workflow and all downstream analysis continue unchanged.

**Two-stage logic (per indent).** The instrument first *finds the surface*
(depth ramps toward the ~2000 nm surface-approach distance while H/E stay `NaN`)
and then performs the *actual indentation* (depth restarts and climbs to the
target with finite, positive H/E). Only the indentation stage is recorded, and a
reading is accepted only when **(a)** the indentation stage has started,
**(b)** hardness and modulus are finite and positive, and **(c)** depth has
reached at least **10 % of the target max depth** (≥ 100 nm for a 1000 nm test).

> This path is meaningful on **real iMicro hardware** (`USE_SIMULATOR = False`),
> where `get_all_ui_state` exposes the live DEPTH/HARDNESS/MODULUS panel. In
> simulator mode those metrics are not on the panel, so the primary download is
> always used. `SCREEN_READ_CONFIG` below exposes the metric keys, screen
> regions, timing, and thresholds so the reader can be recalibrated for a
> different window size or resolution without touching the logic.


In [ ]:
# ============================================================================
# 12.1  SCREEN-READING FALLBACK — configuration + live reader + safe executor
# ----------------------------------------------------------------------------
# PRIMARY source stays the exported result-file download. This fallback runs the
# array ONCE, screen-reads DEPTH/HARDNESS/MODULUS as fast as the server answers,
# then tries the normal export/download. If the download is valid it is used
# (primary); otherwise the live readings are repackaged into the same
# indent_data shape. Each indent's H/E = MEAN of its accepted live samples
# (the panel shows dynamic values during loading and NaN at peak, so averaging
# the accepted ramp samples is the best on-screen estimate).
# ============================================================================
import math
import numpy as np
from datetime import datetime

# --- Configurable screen source / thresholds (recalibrate here, not in code) ---
SCREEN_READ_CONFIG = {
    # (1) Which server UI-state keys hold each live metric (first match wins).
    "metric_keys": {
        "depth":    ["DEPTH", "Depth", "Displacement"],   # nm
        "load":     ["LOAD", "Load", "Force"],            # mN
        "hardness": ["HARDNESS", "Hardness"],             # GPa
        "modulus":  ["MODULUS", "Modulus"],               # GPa
    },
    # (2) OPTIONAL pixel regions (x, y, w, h) for an image-OCR reader on a new
    #     window size / resolution. Unused by the default get_all_ui_state path.
    "screen_regions": {
        "depth":    (150, 800, 210, 90),
        "load":     (150, 900, 210, 90),   # <-- recalibrate to the on-screen LOAD box
        "hardness": (150, 950, 210, 90),
        "modulus":  (150, 1055, 210, 90),
    },
    # (3) Timing — the real ceiling is the get_all_ui_state round-trip latency.
    #     sample_interval_s = pause BETWEEN screen reads (0.0 = as fast as the
    #     server answers). ocr_interval_s = how often to poll the SLOW completion
    #     OCR (kept separate so screen reads run back-to-back in between).
    "sample_interval_s": 0.0,     # 0.0 = read as fast as get_all_ui_state answers (no pause)
    "ocr_interval_s":    5.0,     # poll the SLOW "Tests: N" completion OCR only this often
                                  #   (higher -> more screen reads per second during one indent;
                                  #    lower  -> tighter per-indent bucketing on multi-indent grids)
    "print_interval_s":  5.0,     # print current values + status every 5 s
    # (4) Two-stage detection + acceptance
    "accept_depth_frac":   0.10,          # accept from 10% of target depth onward
    "depth_reset_drop_nm": 400.0,         # a large depth drop => approach -> indent
    "hardness_valid_range": (0.3, 100.0), # was (1.5, 60) — soft phases (~1.4 GPa) are real
    # (5) Output
    "csv_dir": str(DOWNLOAD_DIR),
}


def _screen_metric(ui_state, aliases):
    """First finite numeric value among alias keys, else NaN."""
    if not isinstance(ui_state, dict):
        return float("nan")
    for k in aliases:
        if k in ui_state:
            try:
                return float(ui_state[k])
            except (TypeError, ValueError):
                continue
    return float("nan")


class LiveScreenRecorder:
    """Aggregate live DEPTH/HARDNESS/MODULUS samples into one AVERAGED record per indent."""

    def __init__(self, plan, config, run_id):
        self.cfg = config
        self.sample = plan["sample_name"]
        self.nx, self.ny = int(plan["grid_nx"]), int(plan["grid_ny"])
        self.spacing = float(plan["spacing_um"])
        self.dest_x, self.dest_y = plan["destination"]
        self.n_indents = self.nx * self.ny            # supports 1x1 .. 5x5
        setup = plan.get("setup") or {}
        self.target_depth = float(setup.get("target depth",
                                  cfg.indent_defaults.setup.get("target depth", 1000)))
        self.accept_depth_nm = self.cfg["accept_depth_frac"] * self.target_depth
        self.h_lo, self.h_hi = self.cfg["hardness_valid_range"]
        self.records = {}          # idx(1-based) -> AVERAGED record dict (one per indent)
        self.samples = []          # every ACCEPTED raw sample (transparency / averaging)
        self._by_idx = {}          # idx -> list[(depth, load, H, E)] accepted samples
        self.started = {}          # idx -> bool: actual indentation stage started
        self.max_depth_seen = {}   # idx -> running max depth (approach detection)
        self.n_samples = 0
        self._last_print = 0.0
        self.csv_path = Path(self.cfg["csv_dir"]) / f"{self.sample}_screenread_{run_id}.csv"

    def grid_position(self, idx):
        """1-based indent number -> (row, col, x_um, y_um). Row-major raster order."""
        i0 = idx - 1
        row, col = i0 // self.nx, i0 % self.nx
        gx = self.dest_x + (col - (self.nx - 1) / 2.0) * self.spacing
        gy = self.dest_y + (row - (self.ny - 1) / 2.0) * self.spacing
        return row, col, round(gx, 2), round(gy, 2)

    @staticmethod
    def classify(depth, hardness, modulus):
        h_ok = math.isfinite(hardness) and hardness > 0
        e_ok = math.isfinite(modulus) and modulus > 0
        return "indentation" if (h_ok and e_ok) else "surface_finding"

    def _acceptable(self, idx, depth, hardness, modulus, stage):
        if not self.started.get(idx, False):
            return False, "indentation stage not started"
        if stage != "indentation":
            return False, "surface-finding (H/E NaN)"
        if not (math.isfinite(hardness) and hardness > 0):
            return False, "H not finite/positive"
        if not (math.isfinite(modulus) and modulus > 0):
            return False, "E not finite/positive"
        if not (math.isfinite(depth) and depth >= self.accept_depth_nm):
            return False, f"depth < {self.accept_depth_nm:.0f} nm"
        return True, "ok"

    def observe(self, test_number, depth, load, hardness, modulus):
        """Fold one live screen sample in; return a short status string."""
        self.n_samples += 1
        idx = min(max(int(test_number) + 1, 1), self.n_indents)   # indent being formed
        stage = self.classify(depth, hardness, modulus)

        prev_max = self.max_depth_seen.get(idx, 0.0)
        if math.isfinite(depth):
            if prev_max - depth >= self.cfg["depth_reset_drop_nm"]:
                self.max_depth_seen[idx] = depth
            else:
                self.max_depth_seen[idx] = max(prev_max, depth)
        if stage == "indentation":
            self.started[idx] = True

        accepted, reason = self._acceptable(idx, depth, hardness, modulus, stage)
        status = "accepted" if accepted else f"rejected ({reason})"
        if accepted:
            self.samples.append({
                "test_number": idx, "Depth (nm)": round(depth, 2), "Load (mN)": round(load, 4),
                "Hardness (GPa)": round(hardness, 4), "Modulus (GPa)": round(modulus, 3),
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            self._by_idx.setdefault(idx, []).append((depth, load, hardness, modulus))
            self._recompute(idx)
            self._flush_csv()          # continuously persist to CSV
        self._maybe_print(idx, depth, hardness, modulus, stage, status)
        return status

    def _recompute(self, idx):
        """Rebuild indent idx's record as the MEAN of its accepted samples."""
        pts = self._by_idx[idx]
        depths = [p[0] for p in pts]; loads = [p[1] for p in pts]
        hs = [p[2] for p in pts]; es = [p[3] for p in pts]
        row, col, gx, gy = self.grid_position(idx)
        self.records[idx] = {
            "sample": self.sample, "test_number": idx,
            "grid_row": row, "grid_col": col, "grid_label": f"r{row + 1}c{col + 1}",
            "X (um)": gx, "Y (um)": gy,
            "Depth (nm)": round(max(depths), 2),                 # deepest penetration reached
            "Load (mN)": round(max(loads), 4),                   # peak load at that depth
            "Hardness (GPa)": round(float(np.mean(hs)), 4),      # mean of accepted samples
            "Modulus (GPa)": round(float(np.mean(es)), 3),
            "n_samples": len(pts),
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "validation_status": "valid", "data_source": "screen_read_fallback",
        }

    def _maybe_print(self, idx, depth, hardness, modulus, stage, status):
        now = time.monotonic()
        if now - self._last_print >= self.cfg["print_interval_s"]:
            self._last_print = now
            n_acc = sum(len(v) for v in self._by_idx.values())
            print(f"  [screen] indent {idx:>2}/{self.n_indents} | "
                  f"D={depth:8.1f} nm  H={hardness:6.2f} GPa  E={modulus:8.1f} GPa | "
                  f"{stage:<15s} | {status:<28s} | samples: {n_acc} over {len(self.records)} indent(s)")

    _CSV_COLS = ["sample", "test_number", "grid_row", "grid_col", "grid_label",
                 "X (um)", "Y (um)", "Depth (nm)", "Load (mN)", "Hardness (GPa)", "Modulus (GPa)",
                 "n_samples", "timestamp", "validation_status", "data_source"]

    def _flush_csv(self):
        rows = [self.records[k] for k in sorted(self.records)]
        pd.DataFrame(rows, columns=self._CSV_COLS).to_csv(self.csv_path, index=False)

    def to_result_rows(self):
        """Repackage AVERAGED per-indent records into the DOWNLOADED-CSV column shape."""
        cols = ["Sample", "Indent", "X (um)", "Y (um)", "Load (mN)", "Hardness (GPa)",
                "Modulus (GPa)", "Contact Depth (nm)", "Region (sim truth)"]
        out = []
        for k in sorted(self.records):
            r = self.records[k]
            out.append({
                "Sample": self.sample, "Indent": r["test_number"],
                "X (um)": r["X (um)"], "Y (um)": r["Y (um)"], "Load (mN)": r.get("Load (mN)"),
                "Hardness (GPa)": r["Hardness (GPa)"], "Modulus (GPa)": r["Modulus (GPa)"],
                "Contact Depth (nm)": r["Depth (nm)"],   # screen depth ~ contact-depth proxy
                "Region (sim truth)": "screen_read",
            })
        return pd.DataFrame(out, columns=cols)


def execute_target_plan_with_screen_fallback(
        toolkit, plan, *, download_dir, review_changes=None, show_frames=True,
        fov_width_um=100.0, fov_height_um=75.0, test_timeout_s=None,
        screen_config=SCREEN_READ_CONFIG, run_id=RUN_ID):
    """
    Run one approved plan EXACTLY ONCE (irreversible indents are never re-run).
    Screen-read DEPTH/HARDNESS/MODULUS as fast as the server answers while the
    array runs, then try the normal export/download. Use the download if valid
    (PRIMARY); otherwise fall back to the averaged live readings — returned in
    the identical ``indent_data`` shape ``ui.execute_target_plan`` produces.
    """
    download_dir = Path(download_dir); download_dir.mkdir(parents=True, exist_ok=True)
    tool_log = []

    def _call(tool, args):
        out = toolkit.call_tool(tool, args)
        tool_log.append({"tool": tool, "arguments": json.dumps(args, default=str)[:220],
                         "ok": out.ok, "elapsed_s": round(getattr(out, "elapsed_s", 0.0) or 0.0, 3),
                         "summary": str(out.result if out.ok else out.error)[:180]})
        if not out.ok:
            raise RuntimeError(f"{tool} failed: {out.error}")
        return out.result

    def _read_screen():
        try:
            st = toolkit.call_tool("get_all_ui_state", {})
            ui_state = st.result if st.ok else {}
            lat = round(getattr(st, "elapsed_s", 0.0) or 0.0, 3)
        except Exception:
            ui_state, lat = {}, float("nan")
        mk = screen_config["metric_keys"]
        return (_screen_metric(ui_state, mk["depth"]),
                _screen_metric(ui_state, mk["load"]),
                _screen_metric(ui_state, mk["hardness"]),
                _screen_metric(ui_state, mk["modulus"]), lat)

    # 1. Stage move (recomputed from live position, exactly like ui.execute_target_plan)
    pos0 = _call("get_xyz_positions", {})
    dest_x, dest_y = plan["destination"]
    dx = dest_x - float(pos0.get("x", 0.0)); dy = dest_y - float(pos0.get("y", 0.0))
    if abs(dx) >= 0.05:
        _call("move", {"amount": round(abs(dx), 2), "direction": "right" if dx > 0 else "left", "t": 2})
    if abs(dy) >= 0.05:
        _call("move", {"amount": round(abs(dy), 2), "direction": "up" if dy > 0 else "down", "t": 2})
    pos = _call("get_xyz_positions", {})

    # 2. Confirmation capture at the target
    cap = _call("capture_video_panel_remote", {})
    image_info = ui.download_capture(toolkit, cap if isinstance(cap, dict) else {}, download_dir)
    if show_frames and image_info.get("local_path"):
        ui.show_capture(image_info, title=f"Confirmation frame — {plan['label']}",
                        note="Frame captured after the approved stage move, before indenting.",
                        fov_width_um=fov_width_um, fov_height_um=fov_height_um)

    # 3. Configure + start the indent array
    nx, ny, sp = plan["grid_nx"], plan["grid_ny"], plan["spacing_um"]
    _call("starting_tests", {"sample_name": plan["sample_name"], "setup": plan.get("setup") or {},
          "indents": {"X Indents": nx, "Y Indents": ny, "X Spacing": sp, "Y Spacing": sp},
          "method_type": plan.get("method_type", "eh"),
          "remove_n": int(plan.get("remove_n", 0))})

    _call("start_test_normal", {"t": 0})

    # 4. Monitor + screen-read. The completion OCR is slow, so poll it only every
    #    ocr_interval_s and screen-read back-to-back in between for max throughput.
    recorder = LiveScreenRecorder(plan, screen_config, run_id)
    expected = nx * ny
    sample_dt = float(screen_config.get("sample_interval_s", 0.25))
    ocr_dt = float(screen_config.get("ocr_interval_s", 2.0))
    timeout_s = float(os.environ.get("NB5_TEST_TIMEOUT_S", test_timeout_s or 90.0 * expected + 120.0))
    deadline = time.monotonic() + timeout_s
    stable_complete, status, n_done, last_ocr = 0, {}, 0, 0.0
    lat_samples = []
    print(f"Screen-read safety net active for '{plan['sample_name']}' "
          f"({nx}x{ny} = {expected} indents; accept depth >= {recorder.accept_depth_nm:.0f} nm; "
          f"sample_dt={sample_dt}s, ocr_dt={ocr_dt}s) -> {recorder.csv_path.name}")
    while True:
        now = time.monotonic()
        fresh_ocr = (not status) or (now - last_ocr >= ocr_dt)
        if fresh_ocr:
            status = _call("get_current_sample_and_test_ocr_passive", {})
            last_ocr = time.monotonic()
            n_done = int(status.get("test_number") or 0)
        depth, load, hardness, modulus, lat = _read_screen()
        if math.isfinite(lat):
            lat_samples.append(lat)
        recorder.observe(n_done, depth, load, hardness, modulus)
        if fresh_ocr:
            is_complete = status.get("status") == "complete" or n_done >= expected
            stable_complete = stable_complete + 1 if is_complete else 0
            if stable_complete >= 2:
                break
        if time.monotonic() >= deadline:
            print(f"WARNING: '{plan['sample_name']}' did not report complete within "
                  f"{timeout_s:.0f}s (last {n_done}/{expected}); using whatever was captured.")
            break
        if sample_dt > 0:
            time.sleep(sample_dt)
    if lat_samples:
        print(f"Screen-read latency: get_all_ui_state ~{np.median(lat_samples)*1000:.0f} ms/read "
              f"(min {min(lat_samples)*1000:.0f}, max {max(lat_samples)*1000:.0f}); "
              f"{recorder.n_samples} reads total, {len(recorder.samples)} accepted.")

    # 5. PRIMARY: export + download the result file. Fall back on any failure.
    data_source = data_path = None
    try:
        export_name = f"{plan['sample_name']}_export"
        pr = _call("process_review_file",
                   {"sample_names": [plan["sample_name"]], "open_dir": "",
                    "current_sample_file_name": plan["sample_name"], "export_dir": "exports",
                    "export_file_name": export_name, "review_changes": review_changes or {},
                    "review_type": plan.get("method_type", "eh"), "export_format": "csv"})
        export_file = pr.get("export_file_name", f"{export_name}.csv")
        dl = toolkit.client.download_file("exports", export_file, local_dir=str(download_dir))
        rows = pd.read_csv(dl["local_path"])
        h, e = rows["Hardness (GPa)"], rows["Modulus (GPa)"]
        valid = (h > recorder.h_lo) & (h < recorder.h_hi)
        if int(valid.sum()) == 0 or not np.isfinite(h[valid].mean()):
            raise RuntimeError("downloaded results contained no valid indents")
        data_source, data_path = "result_file_download", dl["local_path"]
        print(f"PRIMARY result-file download OK: {int(valid.sum())}/{len(rows)} valid indents.")
    except Exception as err:
        print(f"PRIMARY result-file download unavailable ({err}) -> using SCREEN-READ fallback.")
        rows = recorder.to_result_rows()
        data_source, data_path = "screen_read_fallback", str(recorder.csv_path)
        if rows.empty:
            print("  screen-read fallback captured NO valid indents — check "
                  "SCREEN_READ_CONFIG['metric_keys'] / thresholds against the live panel.")
        h, e = rows["Hardness (GPa)"], rows["Modulus (GPa)"]
        valid = (h > recorder.h_lo) & (h < recorder.h_hi)

    # 6. Build indent_data in the SAME shape the agent already consumes.
    hv, ev = h[valid], e[valid]
    indent_data = {
        "test_name": plan["sample_name"], "sample_name": plan["sample_name"],
        "n_indents": int(len(rows)), "grid_nx": nx, "grid_ny": ny, "spacing_um": sp,
        "method_type": plan.get("method_type", "eh"),
        "hardness_mean": float(hv.mean()) if len(hv) else float("nan"),
        "hardness_std":  float(hv.std(ddof=0)) if len(hv) else float("nan"),
        "modulus_mean":  float(ev.mean()) if len(ev) else float("nan"),
        "modulus_std":   float(ev.std(ddof=0)) if len(ev) else float("nan"),
        "valid_indent_count": int(valid.sum()),
        "data_path": data_path,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "position": (float(pos.get("x", 0.0)), float(pos.get("y", 0.0))),
        "area_name": plan["label"], "results": rows.to_dict("records"),
        "data_source": data_source, "screen_csv": str(recorder.csv_path),
        "screen_valid_count": len(recorder.records), "n_screen_samples": recorder.n_samples,
    }
    return {"indent_data": indent_data, "rows": rows, "image": image_info,
            "tool_log": tool_log, "monitor_status": status, "recorder": recorder}


def per_round_image_analysis(image_info, round_no, tag=""):
    """Re-run the Section-6 image workflow on the frame captured at this round's site.

    Mirrors the round-1 Image Analysis step (classical CV + optional vision LLM) so
    every round follows the same workflow. Site selection stays in the original FOV
    frame (BO optimizes location only), so this adds per-site context + logging; it
    does not change the BO coordinate frame. Safe to call in manual or autonomous mode.
    """
    img_path = ui.resolve_image_path(image_info) if image_info else None
    if not img_path:
        print(f"  [round {round_no}] no site image to analyze — skipping image analysis.")
        return None
    ui.agent_banner("Image Analysis Agent", f"round {round_no}: segmenting the site frame")
    try:
        cv_result = ui.classical_cv_analysis(img_path, FOV_W, FOV_H)
    except Exception as e:
        print(f"  [round {round_no}] classical CV failed: {e}")
        return None
    vision_result = None
    if TRY_VISION_LLM:
        try:
            from nanoindentation_agents.image_analysis.vision_agent import analyze_image_with_vision
            try:
                _directive = briefings.get("image_analysis", "")          # global from Section 3
            except NameError:
                _directive = state.get("orchestrator_briefings", {}).get("image_analysis", "")
            vision_result = ui.run_with_timeout(
                lambda: analyze_image_with_vision(
                    str(img_path), research_question=state["research_question"],
                    experiment_context=state["experiment_context"]
                        + "\n\n[ORCHESTRATOR DIRECTIVE]\n" + _directive, cfg=cfg),
                seconds=120, label=f"Vision LLM (round {round_no})")
        except Exception as e:
            print(f"  [round {round_no}] vision LLM skipped: {e}")
    analysis, provenance = ui.combine_image_analysis(cv_result, vision_result)
    try:
        state["image_history"].append(image_info)
    except Exception:
        pass
    msg_log.log("Image Analysis Agent", "Orchestrator Agent",
                f"Round {round_no} ({tag}): site image analyzed [{provenance}]",
                detail=str(analysis)[:200])
    return analysis


print("Screen-reading fallback ready. Primary source = result-file download; "
      "fallback = live get_all_ui_state (DEPTH/HARDNESS/MODULUS), averaged per indent.")


In [ ]:
print(toolkit.call_tool("get_xyz_positions", {}))
print(toolkit.call_tool("get_all_ui_state", {}))


In [ ]:
indent_gate_1.require_approved()          # <-- hard stop if Gate 2 was not approved
ui.agent_banner("Instrument Executor", "executing the approved round-1 plans")

def record_execution(res, plan, gate):
    """Append one executed array to state + message log (used in every round)."""
    executions.append(res)
    d = res["indent_data"]
    src = d.get("data_source", "result_file_download")
    state["indent_history"].append(d)
    state["image_history"].append(res["image"])
    state["visited_target_keys"].append(target_visit_key(plan["target"]))
    state["current_position"] = d["position"]
    state["decision_history"].append({
        "cycle": len(state["decision_history"]) + 1, "decision_type": "indent_array",
        "tool_name": "starting_tests/start_test_normal", "approved_via": gate.gate_id,
        "arguments": {"sample": d["test_name"], "grid": f"{d['grid_nx']}x{d['grid_ny']}",
                      "data_source": src},
        "result_summary": f"H={d['hardness_mean']:.2f}±{d['hardness_std']:.2f} GPa, "
                          f"E={d['modulus_mean']:.1f} GPa ({d['valid_indent_count']} valid) "
                          f"[source: {src}]",
    })
    msg_log.log("Instrument Executor", "Measurement Critic",
                f"{d['area_name']}: H = {d['hardness_mean']:.2f} ± {d['hardness_std']:.2f} GPa, "
                f"E = {d['modulus_mean']:.1f} ± {d['modulus_std']:.1f} GPa "
                f"({d['valid_indent_count']}/{d['n_indents']} valid indents) [source: {src}]",
                detail=f"data: {d['data_path']} | source: {src}")
    return d

for plan in round1_plans:
    # Primary = result-file download; automatic screen-read fallback if that fails.
    res = execute_target_plan_with_screen_fallback(
        toolkit, plan, download_dir=DOWNLOAD_DIR,
        review_changes=dict(cfg.indent_defaults.review_changes),
        show_frames=True, fov_width_um=FOV_W, fov_height_um=FOV_H,
    )
    record_execution(res, plan, indent_gate_1)

display(ui.indent_results_table(state["indent_history"]))
for ex in executions:
    if ex["rows"] is not None and not ex["rows"].empty:
        ui.plot_load_depth(ex["rows"], title=f"Load–depth — {ex['indent_data']['area_name']}")
    else:
        print(f"No indent rows to plot for {ex['indent_data']['area_name']} "
              f"(source: {ex['indent_data'].get('data_source')}).")


### 12.2  Round-1 result summary — source, validity, and dispersion

Per-array and per-indent view of round 1: which **data source** produced each
array (result-file download vs screen-read fallback), how many indents are
**valid vs missing**, and the **mean / std / min / max / CV%** of hardness and
modulus. Run this after the executor cell above.


In [ ]:
# Round-1 summary: each test result, data source, valid/missing, and dispersion stats.
def _cv_pct(series):
    m = series.mean()
    return float(series.std(ddof=0) / m * 100.0) if len(series) and m else float("nan")

_H_LO, _H_HI = SCREEN_READ_CONFIG["hardness_valid_range"]
summary_rows, per_test_rows, sample_rows = [], [], []
for ex in executions:
    d = ex["indent_data"]
    src = d.get("data_source", "result_file_download")
    df = ex.get("rows")
    expected = d["grid_nx"] * d["grid_ny"]
    if df is not None and not df.empty and "Hardness (GPa)" in df:
        h, e = df["Hardness (GPa)"], df["Modulus (GPa)"]
        valid = (h > _H_LO) & (h < _H_HI)
        hv, ev = h[valid], e[valid]
        n_valid, n_recorded = int(valid.sum()), int(len(df))
        for _, r in df.iterrows():
            hh = float(r["Hardness (GPa)"])
            per_test_rows.append({
                "array": d["test_name"], "source": src, "indent": r.get("Indent", ""),
                "X (um)": r.get("X (um)"), "Y (um)": r.get("Y (um)"),
                "H (GPa)": round(hh, 2), "E (GPa)": round(float(r["Modulus (GPa)"]), 1),
                "depth (nm)": round(float(r.get("Contact Depth (nm)", float("nan"))), 1),
                "valid": bool(_H_LO < hh < _H_HI),
            })
    else:
        hv = ev = pd.Series(dtype=float)
        n_valid, n_recorded = 0, int(d.get("n_indents", 0))

    # accepted raw screen samples that the per-indent averages were built from
    rec = ex.get("recorder")
    if rec is not None:
        for s in getattr(rec, "samples", []):
            sample_rows.append({"array": d["test_name"], **s})

    summary_rows.append({
        "array": d["test_name"], "region": d.get("area_name", ""), "data_source": src,
        "expected": expected, "recorded": n_recorded,
        "valid": n_valid, "missing": expected - n_valid,
        "H mean": round(hv.mean(), 2) if len(hv) else float("nan"),
        "H std":  round(hv.std(ddof=0), 2) if len(hv) else float("nan"),
        "H min":  round(hv.min(), 2) if len(hv) else float("nan"),
        "H max":  round(hv.max(), 2) if len(hv) else float("nan"),
        "H CV%":  round(_cv_pct(hv), 1) if len(hv) else float("nan"),
        "E mean": round(ev.mean(), 1) if len(ev) else float("nan"),
        "E std":  round(ev.std(ddof=0), 1) if len(ev) else float("nan"),
        "E min":  round(ev.min(), 1) if len(ev) else float("nan"),
        "E max":  round(ev.max(), 1) if len(ev) else float("nan"),
        "E CV%":  round(_cv_pct(ev), 1) if len(ev) else float("nan"),
    })

print("=== Round 1 — per-array summary (data source, validity, dispersion) ===")
display(pd.DataFrame(summary_rows))
print("=== Round 1 — per-indent results (screen-read H/E = mean of accepted samples) ===")
display(pd.DataFrame(per_test_rows))
if sample_rows:
    print("=== Accepted raw screen samples (the values each per-indent average is built from) ===")
    _sdf = pd.DataFrame(sample_rows)
    display(_sdf)
    _agg = (_sdf.groupby(["array", "test_number"])[["Hardness (GPa)", "Modulus (GPa)"]]
                 .agg(["count", "mean"]).round(3))
    print("Per-indent average of accepted samples:")
    display(_agg)

_sources = sorted({ex["indent_data"].get("data_source") for ex in executions if ex["indent_data"].get("data_source")})
print(f"\nData source(s) in use this round: {_sources}")
if "screen_read_fallback" in _sources:
    print("NOTE: at least one array used the SCREEN-READING FALLBACK — the "
          "result-file download was unavailable, so live-panel readings were used.")


## 13. 📊 Measurement Critic — objective-derived success criteria

The critic's thresholds are no longer hardcoded: `objective_criteria` in state (written by the
orchestrator at Gate 0) supplies the evidence minimums, the required improvement margin, and the
**property weights/directions** used to score arrays. The rationale is tagged
`[OBJECTIVE CRITERIA]` so you can audit which rule set produced the verdict.

In [ ]:
ui.agent_banner("Measurement Critic", "evaluating round-1 evidence against the OBJECTIVE criteria")

print("Criteria in force:", json.dumps(
    {k: v for k, v in state["objective_criteria"].items() if k != "objective_summary"}, indent=2))

state.update(measurement_critic_node(state))
routing = get_routing_decision(state)
ui.critic_panel("Measurement Critic verdict", state["measurement_critique"], approved=None)
print(f"Critic decision:  {state['measurement_critique_decision']!r}")
msg_log.log("Measurement Critic", "Orchestrator Agent",
            f"Decision: {state['measurement_critique_decision']} (objective criteria)",
            detail=state["measurement_critique"][:250])

## 14. 🎯 THE ORCHESTRATOR ROUND LOOP — re-runnable cells A → D

This block replaces notebook 3's one-shot supervisor. **Run cells A→D in order, then go back to
Cell A for the next round** — each pass is one adaptive round:

| Cell | What happens |
|---|---|
| **A** | Orchestrator fits the GP surrogate, scores progress against the objective, and **decides**: `continue`, `refine_hypothesis`, `objective_met`, `budget_exhausted`, or `escalate_to_human` |
| **B** | Acquisition Agent proposes the next site(s) — **Expected Improvement maps shown** — and the Planning Agent builds the gated action plan |
| **C** | 🛑 Your approval (one gate per round, auto-approved in unattended mode) |
| **D** | Execute, re-run the Measurement Critic, record the round, advance |

Cell A is **idempotent** (pure evaluation — safe to re-run anytime). Cell D refuses to
double-execute an already-executed round. When the orchestrator says anything other than
`continue`, skip to **section 15 (results)** — or, for `refine_hypothesis`, re-run section 8
(the critic's feedback is already in state) and come back to Cell A.

*Hands-free alternatives:* the single cell after D runs **all remaining rounds by itself**,
stopping at `objective_met`, the indent budget, or `MAX_ROUNDS` — whichever comes first. It
engages when either switch in section 1 is on:

- `AUTO_RUN_ROUNDS = True` — you still approve Gate 0/1/2 manually; only the adaptive round
  gates auto-approve (recommended);
- `AUTO_APPROVE_ALL = True` — every gate in the notebook auto-approves (used by headless runs).

**Every round runs the same workflow as round 1:** per-round image analysis → execute with the *same dual retrieval* (result-file download **or** screen-read fallback) → save under a new name → screen+validate → objective check.

**Autonomous option:** set `RUN_AUTONOMOUS_CAMPAIGN = True` in Section 1 to let the **AUTONOMOUS / UNATTENDED CAMPAIGN** cell below run *all* remaining rounds hands-free on real hardware (auto-approves each irreversible round; stops at budget / MAX_ROUNDS / objective). Otherwise use cells A–D, one pass per round.


In [ ]:
# ============================ CELL A — evaluate & decide (idempotent) ============================
ui.agent_banner("Orchestrator Agent", "evaluating progress toward the objective")

surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
progress = orch.evaluate_progress(state, surrogate)
decision = orch.decide_next_action(state, progress, surrogate)
round_no = progress.round_index + 1

display(progress.table())
best = progress.best_site
if best:
    print(f"Best site so far: ({best['x_um']:.1f}, {best['y_um']:.1f}) µm in '{best['array']}' — "
          f"H={best['hardness_gpa']:.2f} GPa, E={best['modulus_gpa']:.1f} GPa "
          f"(objective score {best['scalar_value']:.2f})")

print(f"\n{'='*76}\nORCHESTRATOR DECISION (round {round_no}):  {decision.action.upper()}"
      f"   [advisor: {decision.advisor}]\n{decision.reason}\n{'='*76}")
if decision.llm_comment and decision.advisor == "rules":
    print(f"LLM advisor concurred: {decision.llm_comment}")

msg_log.log("Orchestrator Agent", "Human Operator",
            f"Round {round_no}: {decision.action} — {decision.reason[:120]}",
            detail=decision.reason)

if decision.action != "continue":
    # record terminal decisions once (idempotent on re-runs of this cell)
    _decs = state.get("orchestrator_decisions") or []
    if not (_decs and _decs[-1].get("round") == progress.round_index
            and _decs[-1].get("action") == decision.action):
        state.update(orch.decision_state_updates(state, decision, progress))
    if decision.action == "refine_hypothesis":
        print("\n>> Re-run section 8 (hypothesis + critic) — the measurement critique is in state —")
        print(">> then come back to Cell A.")
    else:
        print(f"\n>> The campaign is over ({decision.action}). Skip to section 15 for results.")

In [ ]:
    # --- Refresh the live frame before proposing (REAL hardware only) ----------------------
    # After the previous round's indents the stage is parked at that round's site, so the frame
    # in state["current_image"] is stale. Re-center on the campaign's FIXED FOV reference
    # (capture_pos) and grab a fresh frame, so the proposal is made against what the indenter is
    # actually looking at now. capture_pos and the BO coordinate frame are deliberately left
    # unchanged — only the picture updates. Uses the SAME move convention as the executor, so if
    # the fresh frame fails to re-center, the direction mapping is confirmed inverted (open issue).
    if not USE_SIMULATOR:
        def _call(tool, args):
            return toolkit.call_tool(tool, args).result
        _p0 = _call("get_xyz_positions", {})
        _dx = capture_pos[0] - float(_p0.get("x", 0.0))
        _dy = capture_pos[1] - float(_p0.get("y", 0.0))
        if abs(_dx) >= 0.05:
            _call("move", {"amount": round(abs(_dx), 2), "direction": "right" if _dx > 0 else "left", "t": 0})
        if abs(_dy) >= 0.05:
            _call("move", {"amount": round(abs(_dy), 2), "direction": "up" if _dy > 0 else "down", "t": 0})
        _here = _call("get_xyz_positions", {})
        print(f"Re-centered on FOV reference: stage now "
              f"({_here.get('x', 0):.1f}, {_here.get('y', 0):.1f}) um  "
              f"(target {capture_pos[0]:.1f}, {capture_pos[1]:.1f} um).")
        _cap = _call("capture_video_panel_remote", {})
        _img = ui.download_capture(toolkit, _cap if isinstance(_cap, dict) else {}, DOWNLOAD_DIR)
        if _img.get("local_path"):
            _img["width_um"] = FOV_W
            _img["height_um"] = FOV_H
            _prev = state.get("current_image") or {}
            if _prev.get("um_per_px") is not None:
                _img["um_per_px"] = _prev["um_per_px"]
            state["current_image"] = _img                       # only the picture changes
            ui.show_capture(_img,
                            title=f"Round {round_no} — fresh frame at FOV reference (pre-proposal)",
                            note="Re-centered on capture_pos and re-imaged before proposing this round.",
                            fov_width_um=FOV_W, fov_height_um=FOV_H)
            # optional: mirror round-1's image analysis on the fresh frame
            per_round_image_analysis(_img, round_no, tag="pre-proposal")
        else:
            print("  [warn] re-capture failed — keeping the previous frame for this round's proposal.")


In [ ]:
# ============================ CELL B — propose sites & open the round gate ============================
if decision.action != "continue":
    print(f"No proposal — the orchestrator decided '{decision.action}'. Skip to section 15.")
else:
    if decision.proposal_source == "bo_acquisition":
        ui.agent_banner("Acquisition Agent (BO)",
                        f"maximizing Expected Improvement over the FOV ({surrogate.mode}, "
                        f"{surrogate.n_points} observations)")
        proposals = surrogate.propose_sites(
            decision.n_sites, round_index=round_no, stage_position=capture_pos,
            fov_width_um=FOV_W, fov_height_um=FOV_H)

        maps = surrogate.posterior_maps(n=60)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
        for ax, key, title, cmap in [
                (axes[0], "scalar_mean", "objective posterior mean", "viridis"),
                (axes[1], "scalar_sigma", "posterior uncertainty (σ)", "magma"),
                (axes[2], "ei", "Expected Improvement", "cividis")]:
            im = ax.imshow(maps[key], origin="lower", extent=maps["extent"], aspect="equal", cmap=cmap)
            ax.scatter(surrogate.X[:, 0], surrogate.X[:, 1], s=14, c="white",
                       edgecolor="k", linewidths=0.4, label="measured")
            for p in proposals:
                ax.scatter([p["x_um"]], [p["y_um"]], marker="*", s=300, c="red",
                           edgecolor="k", linewidths=0.6, label="proposed")
            ax.set_title(title); ax.set_xlabel("stage x (µm)"); ax.set_ylabel("stage y (µm)")
            fig.colorbar(im, ax=ax, shrink=0.85)
        handles, labels = axes[0].get_legend_handles_labels()
        by = dict(zip(labels, handles)); axes[0].legend(by.values(), by.keys(), loc="lower left", fontsize=8)
        fig.suptitle(f"Round {round_no} — GP posterior + acquisition ({surrogate.mode})", y=1.03)
        plt.tight_layout(); plt.show()
    else:
        ui.agent_banner("Orchestrator Agent", "selecting the next unvisited objective-ranked image target")
        unvisited = [t for t in state["measurement_plan"]
                     if target_visit_key(t) not in state["visited_target_keys"]]
        proposals = [dict(t) for t in unvisited[:decision.n_sites]]

    assert proposals, "No proposable sites left — re-run Cell A (the orchestrator will stop the campaign)."
    display(pd.DataFrame([{
        "site": p["label"], "x (µm)": p["x_um"], "y (µm)": p["y_um"],
        "why": p.get("reason", p.get("rank_reason", ""))[:100],
    } for p in proposals]))

    round_plans = []
    for p in proposals:
        plan = ui.plan_target_actions(
            p, capture_pos, sample_name=f"a{round_no}_{p['label']}_{RUN_ID}",
            grid_nx=objective.constraints.grid_nx, grid_ny=objective.constraints.grid_ny,
            spacing_um=objective.constraints.spacing_um,
            setup=dict(cfg.indent_defaults.setup), method_type=cfg.indent_defaults.method_type,
            fov_width_um=FOV_W, fov_height_um=FOV_H,
            expected_outcome=p.get("reason", "objective-driven site"),
        )
        round_plans.append(plan)
    ui.plot_planned_grids(state["current_image"], round_plans, capture_pos,
                          title=f"Adaptive round {round_no} — proposed by {decision.proposal_source}",
                          fov_width_um=FOV_W, fov_height_um=FOV_H)

    _cost = sum(p["grid_nx"] * p["grid_ny"] for p in round_plans)
    print(f"Budget: {_cost} indents this round; {progress.budget_remaining} remaining of "
          f"{objective.constraints.indent_budget}")

    state["acquisition_meta"] = {**surrogate.summary(), "round": round_no,
                                 "proposal_source": decision.proposal_source,
                                 "proposals": proposals}
    msg_log.log("Acquisition Agent (BO)" if decision.proposal_source == "bo_acquisition"
                else "Orchestrator Agent",
                "Planning Agent",
                f"Round {round_no}: {len(proposals)} site(s) proposed via {decision.proposal_source}",
                detail="; ".join(p.get("reason", "")[:100] for p in proposals))

    # one gate per round; re-running this cell replaces an undecided gate with the same id
    _gid = f"gate-adaptive-round-{round_no}"
    ui.ApprovalGate.registry[:] = [g for g in ui.ApprovalGate.registry
                                   if not (g.gate_id == _gid and g.decision is None)]
    round_gate = ui.ApprovalGate(
        _gid, f"Adaptive round {round_no} — {decision.proposal_source} site(s)",
        description=(f"Orchestrator: {decision.reason[:220]}\n"
                     f"Approving executes {_cost} indents at {len(round_plans)} location(s)."),
        action_cards=[card for p in round_plans for card in p["cards"]],
    )

In [ ]:
# ============================ CELL C — your approval for this round ============================
# ==> YOUR DECISION IS REQUIRED HERE (only when the orchestrator said "continue") <==
#     MY_DECISION = "approve"       or       MY_DECISION = "reject: <your reason>"
MY_DECISION = "approve"

if decision.action != "continue":
    print(f"No gate this round — the orchestrator decided '{decision.action}'.")
elif AUTO_APPROVE_ALL:
    round_gate.approve(approved_by="AUTO (unattended run)",
        note="Auto-approved: AUTO_APPROVE_ALL is enabled in section 1")
elif MY_DECISION.strip().lower().startswith("approve"):
    round_gate.approve(approved_by=OPERATOR_NAME,
        note=MY_DECISION.partition(":")[2].strip() or "Approved by operator")
elif MY_DECISION.strip().lower().startswith("reject"):
    round_gate.reject(reason=MY_DECISION.partition(":")[2].strip() or "Rejected by operator",
        approved_by=OPERATOR_NAME)
else:
    print("PAUSED — this round's gate is still PENDING.")
    print('  Edit this cell: set MY_DECISION = "approve" or "reject: <reason>", then re-run it.')

if decision.action == "continue" and round_gate.decision:
    _approver = "AUTO Approver" if AUTO_APPROVE_ALL else "Human Operator"
    msg_log.log(_approver, "Instrument Executor",
                f"Round {round_no} APPROVED" if round_gate.is_approved
                else f"Round {round_no} REJECTED — nothing will execute",
                detail=round_gate.note)

In [ ]:
# ===================== CELL C½ — MOVE per the agent's Cell C suggestion, then CONFIRM =====================
# Executes ONLY the agent's proposed move (plan["moves"] — the values shown on the Cell C card),
# from the stage's current position, then captures a confirmation frame to verify before Cell D.
if decision.action != "continue":
    print(f"No move — orchestrator decided '{decision.action}'.")
else:
    round_gate.require_approved()

    def _call(tool, args):
        out = toolkit.call_tool(tool, args)
        if not out.ok:
            raise RuntimeError(f"{tool} failed: {out.error}")
        return out.result

    plan = round_plans[0]
    dest_x, dest_y = plan["destination"]

    print(f"Agent's proposed move for '{plan['label']}':")
    if not plan["moves"]:
        print("   (site is at the photo center — no move)")
    for mv in plan["moves"]:
        print(f"   move {mv['amount']} um {mv['direction']}")
        _call("move", {"amount": round(float(mv["amount"]), 2), "direction": mv["direction"], "t": 2})

    p1 = _call("get_xyz_positions", {})
    err = ((float(p1["x"]) - dest_x) ** 2 + (float(p1["y"]) - dest_y) ** 2) ** 0.5
    print(f"\nLanded at ({float(p1['x']):.1f}, {float(p1['y']):.1f}) um  |  target ({dest_x:.1f}, {dest_y:.1f})  "
          f"|  off by {err:.2f} um")

    cap = _call("capture_video_panel_remote", {})
    conf = ui.download_capture(toolkit, cap if isinstance(cap, dict) else {}, DOWNLOAD_DIR)
    if conf.get("local_path"):
        conf["width_um"] = FOV_W; conf["height_um"] = FOV_H
        ui.show_capture(conf, title=f"Confirmation — '{plan['label']}' (round {round_no})",
                        note="Crosshair should sit on the intended feature. Verify BEFORE Cell D.",
                        fov_width_um=FOV_W, fov_height_um=FOV_H)
    print("\n>> On the right feature?  Run CELL D to indent.  If not, STOP.")


In [ ]:
# ==================== CELL D — clear old test, run the round, re-critique, advance ====================
if decision.action != "continue":
    print(f"Nothing to execute — the orchestrator decided '{decision.action}'.")
else:
    round_gate.require_approved()
    if any(dd.get("approved_via") == round_gate.gate_id for dd in state["decision_history"]):
        print(f"Round {round_no} was already executed — re-run Cell A to start round {round_no + 1}.")
    else:
        ui.agent_banner("Instrument Executor", f"executing adaptive round {round_no}")
        _in_list = state.get("imicro_queue_count", len(state["indent_history"]))   # samples still queued
        for i, plan in enumerate(round_plans):
            plan["remove_n"] = _in_list if i == 0 else 0        # remove old test(s) before adding this one
            res = execute_target_plan_with_screen_fallback(     # Cell 1 already moved -> this skips its move
                toolkit, plan, download_dir=DOWNLOAD_DIR,
                review_changes=dict(cfg.indent_defaults.review_changes),
                show_frames=True, fov_width_um=FOV_W, fov_height_um=FOV_H,
            )
            per_round_image_analysis(res["image"], round_no, tag="manual")
            record_execution(res, plan, round_gate)
        state["imicro_queue_count"] = len(round_plans)          # only this round's sample(s) remain queued
        display(ui.indent_results_table(state["indent_history"]))

        ui.agent_banner("Measurement Critic", f"re-evaluating with round-{round_no} evidence")
        state.update(measurement_critic_node(state))
        ui.critic_panel(f"Measurement Critic — after round {round_no}",
                        state["measurement_critique"],
                        approved=state["measurement_critique_decision"] == "hypothesis_confirmed")
        msg_log.log("Measurement Critic", "Orchestrator Agent",
                    f"Decision: {state['measurement_critique_decision']} (objective criteria)",
                    detail=state["measurement_critique"][:250])

        state.update(orch.decision_state_updates(state, decision, progress))
        state["orchestrator_round"] = round_no
        print(f"\nRound {round_no} complete — re-run CELL A to let the orchestrator decide round {round_no + 1}.")


In [ ]:
# ==================== SECTION 14 — AUTONOMOUS / UNATTENDED CAMPAIGN ====================
# Hands-free adaptive rounds. Each round: decide -> BO site -> auto-approve -> MOVE (verified) ->
# clear old test -> indent (dual retrieval) -> image analysis -> critic -> advance.
# Stops at objective_met / INDENT_BUDGET / MAX_ROUNDS, OR aborts if a stage move lands off target.
AUTO_MOVE_TOL_UM = 3.0        # abort the campaign if a move ends farther than this from the target

_auto_on = AUTO_APPROVE_ALL or AUTO_RUN_ROUNDS or RUN_AUTONOMOUS_CAMPAIGN
if not _auto_on:
    print("Manual mode — re-run cells A–D above, one pass per round.")
    print("For a hands-free run: set RUN_AUTONOMOUS_CAMPAIGN = True (real hardware) or "
          "AUTO_RUN_ROUNDS = True (simulator) in Section 1, re-run that cell, then this one.")
else:
    if RUN_AUTONOMOUS_CAMPAIGN and not USE_SIMULATOR:
        print("⚠️  AUTONOMOUS REAL-HARDWARE run — auto-approving every irreversible round "
              "until budget / MAX_ROUNDS / objective (or a bad move aborts it).\n")

    def _call(tool, args):
        out = toolkit.call_tool(tool, args)
        if not out.ok:
            raise RuntimeError(f"{tool} failed: {out.error}")
        return out.result

    decision = None
    for _pass in range(MAX_ROUNDS + 2):                      # hard safety cap
        # (A) evaluate progress + decide
        surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
        progress = orch.evaluate_progress(state, surrogate)
        decision = orch.decide_next_action(state, progress, surrogate)
        round_no = progress.round_index + 1
        print(f"\n=== AUTO round {round_no}: {decision.action} — {decision.reason[:110]}")
        msg_log.log("Orchestrator Agent", "Human Operator",
                    f"Round {round_no}: {decision.action} — {decision.reason[:120]}",
                    detail=decision.reason)

        if decision.action != "continue":
            _decs = state.get("orchestrator_decisions") or []
            if not (_decs and _decs[-1].get("round") == progress.round_index
                    and _decs[-1].get("action") == decision.action):
                state.update(orch.decision_state_updates(state, decision, progress))
            break

        # (B) optimize the next SITE via BO / ranked targets
        if decision.proposal_source == "bo_acquisition":
            proposals = surrogate.propose_sites(
                decision.n_sites, round_index=round_no, stage_position=capture_pos,
                fov_width_um=FOV_W, fov_height_um=FOV_H)
        else:
            unvisited = [t for t in state["measurement_plan"]
                         if target_visit_key(t) not in state["visited_target_keys"]]
            proposals = [dict(t) for t in unvisited[:decision.n_sites]]
        if not proposals:
            print("No proposable sites left — stopping.")
            break

        round_plans = [ui.plan_target_actions(
            p, capture_pos, sample_name=f"a{round_no}_{p['label']}_{RUN_ID}",
            grid_nx=objective.constraints.grid_nx, grid_ny=objective.constraints.grid_ny,
            spacing_um=objective.constraints.spacing_um,
            setup=dict(cfg.indent_defaults.setup), method_type=cfg.indent_defaults.method_type,
            fov_width_um=FOV_W, fov_height_um=FOV_H,
            expected_outcome=p.get("reason", "objective-driven site"),
        ) for p in proposals]

        # (C) auto-approve this round's gate
        _gid = f"gate-adaptive-round-{round_no}"
        ui.ApprovalGate.registry[:] = [g for g in ui.ApprovalGate.registry
                                       if not (g.gate_id == _gid and g.decision is None)]
        round_gate = ui.ApprovalGate(
            _gid, f"Adaptive round {round_no} — {decision.proposal_source} site(s)",
            description=decision.reason[:220],
            action_cards=[card for p in round_plans for card in p["cards"]])
        _who = ("AUTONOMOUS (real hardware)" if (RUN_AUTONOMOUS_CAMPAIGN and not USE_SIMULATOR)
                else "AUTO (unattended run)")
        round_gate.approve(approved_by=_who,
                           note="Auto-approved: hands-free campaign enabled in Section 1")

        # (D) MOVE (verified) -> clear old test -> execute
        _in_list = int(state.get("imicro_queue_count", len(state["indent_history"])))
        _abort = False
        try:
            for i, plan in enumerate(round_plans):
                # move to the ABSOLUTE destination from wherever the stage is (robust across rounds)
                p0 = _call("get_xyz_positions", {})
                dx = plan["destination"][0] - float(p0["x"]); dy = plan["destination"][1] - float(p0["y"])
                if abs(dx) >= 0.05:
                    _call("move", {"amount": round(abs(dx), 2), "direction": "right" if dx > 0 else "left", "t": 2})
                if abs(dy) >= 0.05:
                    _call("move", {"amount": round(abs(dy), 2), "direction": "up" if dy > 0 else "down", "t": 2})
                p1 = _call("get_xyz_positions", {})
                err = ((float(p1["x"]) - plan["destination"][0])**2 + (float(p1["y"]) - plan["destination"][1])**2) ** 0.5
                print(f"    move -> ({float(p1['x']):.1f}, {float(p1['y']):.1f}) um  |  off target {err:.2f} um")
                if err > AUTO_MOVE_TOL_UM:
                    print(f"    ⚠️  OFF TARGET by {err:.1f} um (> {AUTO_MOVE_TOL_UM}) — ABORTING, NOT indenting.")
                    _abort = True
                    break

                plan["remove_n"] = _in_list if i == 0 else 0        # clear old queued test(s) before adding
                res = execute_target_plan_with_screen_fallback(     # move already done -> executor skips it
                    toolkit, plan, download_dir=DOWNLOAD_DIR,
                    review_changes=dict(cfg.indent_defaults.review_changes),
                    show_frames=False, fov_width_um=FOV_W, fov_height_um=FOV_H,
                )
                per_round_image_analysis(res["image"], round_no, tag=decision.proposal_source)
                d = record_execution(res, plan, round_gate)
                print(f"    {d['area_name']}: H={d['hardness_mean']:.2f} GPa, "
                      f"E={d['modulus_mean']:.1f} GPa  [source: {d.get('data_source')}]")
        except Exception as e:
            print(f"    ⚠️  round {round_no} failed ({type(e).__name__}: {e}) — stopping campaign.")
            _abort = True

        if _abort:
            break
        state["imicro_queue_count"] = len(round_plans)          # only this round's sample(s) remain queued

        # (E) screen + validate this round's evidence
        state.update(measurement_critic_node(state))
        msg_log.log("Measurement Critic", "Orchestrator Agent",
                    f"Decision: {state['measurement_critique_decision']} (objective criteria)",
                    detail=state["measurement_critique"][:250])
        state["acquisition_meta"] = {**surrogate.summary(), "round": round_no,
                                     "proposal_source": decision.proposal_source,
                                     "proposals": proposals}
        state.update(orch.decision_state_updates(state, decision, progress))
        state["orchestrator_round"] = round_no

    print(f"\nCampaign finished: {decision.action if decision else 'no rounds'} after "
          f"{state.get('orchestrator_round', 0)} adaptive round(s). See Section 15 for results.")


## 15. Results — property maps, GP posterior, objective attainment

Everything measured, plus the two artifacts unique to the objective-driven design: the **GP
posterior over the whole field of view** (what the orchestrator *believes* about unmeasured
regions) and the **objective attainment report** (each success criterion, met or not).

In [ ]:
# =====================================================================
#  STATISTICAL ANALYSIS + JOURNAL-READY RESULTS NARRATIVE
#  Auto-computed from all_rows; safe if SciPy is absent. Self-contained.
# =====================================================================
import numpy as np, pandas as pd
try:
    from scipy import stats as _sps
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

all_rows = globals().get("all_rows")
if all_rows is None:                       # cell can run on its own now
    all_rows = ui.combined_rows(executions)
LIT_EXPECTATIONS = globals().get("LIT_EXPECTATIONS", {})

df = all_rows.copy()
H, E = "Hardness (GPa)", "Modulus (GPa)"

# --- self-contained: rebuild region labels if the figure cell didn't run ---
regions  = globals().get("regions")  or list(dict.fromkeys(df["region"]))
label_of = globals().get("label_of") or {r: f"Test {i}" for i, r in enumerate(regions, 1)}

def _cohen_d(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = len(a), len(b)
    if na < 2 or nb < 2: return np.nan
    sp = np.sqrt(((na - 1) * a.var(ddof=1) + (nb - 1) * b.var(ddof=1)) / (na + nb - 2))
    return (a.mean() - b.mean()) / sp if sp else np.nan


In [ ]:
# =====================================================================
#  PUBLICATION-QUALITY RESULTS FIGURES  (self-contained; drop-in cell)
#  Requires: all_rows (from ui.combined_rows) and LIT_EXPECTATIONS
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

plt.rcParams.update({
    "figure.dpi":        120,
    "savefig.dpi":       600,
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.titleweight":  "bold",
    "axes.labelsize":    10.5,
    "axes.linewidth":    1.0,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "legend.frameon":    False,
    "figure.facecolor":  "white",
})

# regions in order of first appearance -> generic Test 1, Test 2, ...
regions = list(dict.fromkeys(all_rows["region"]))
label_of = {r: f"Test {i}" for i, r in enumerate(regions, 1)}

palette = plt.get_cmap("tab10")
RCOLOR = {r: palette(i) for i, r in enumerate(regions)}


# =====================================================================
#  FIGURE 1 — Spatial property maps (relative coords, no giant offset)
# =====================================================================
def fig_property_map(df):
    x = df["X (um)"] - df["X (um)"].min()          # relative -> no 1.6e4 offset
    y = df["Y (um)"] - df["Y (um)"].min()
    fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.7), constrained_layout=True)
    specs = [("Hardness (GPa)", "inferno", "$H$ (GPa)"),
             ("Modulus (GPa)",  "viridis", "$E_r$ (GPa)")]
    for ax, (col, cmap, cbar_lbl), tag in zip(axes, specs, "ab"):
        sc = ax.scatter(x, y, c=df[col], cmap=cmap,
                        s=70, edgecolor="k", linewidths=0.5, zorder=3)
        cb = fig.colorbar(sc, ax=ax, pad=0.02, fraction=0.05)
        cb.set_label(cbar_lbl)
        ax.set_xlabel("Relative x (µm)")
        ax.set_ylabel("Relative y (µm)")
        ax.set_title(f"({tag}) {col.split()[0]} map")
        ax.margins(0.18)
    fig.suptitle("Spatial map of measured properties (one marker = one indent)",
                 fontsize=11, fontweight="bold")
    fig.savefig("fig_property_map.png", bbox_inches="tight")
    plt.show()


# =====================================================================
#  FIGURE 2 — Property-space map: H vs E, coloured by test (key figure)
# =====================================================================
def fig_property_space(df):
    fig, ax = plt.subplots(figsize=(5.6, 4.6), constrained_layout=True)
    for r in regions:
        g = df[df["region"] == r]
        ax.scatter(g["Modulus (GPa)"], g["Hardness (GPa)"],
                   s=55, color=RCOLOR[r], edgecolor="k", linewidths=0.5,
                   alpha=0.9, label=label_of[r], zorder=3)
        ax.scatter(g["Modulus (GPa)"].mean(), g["Hardness (GPa)"].mean(),
                   marker="X", s=170, color=RCOLOR[r],
                   edgecolor="k", linewidths=1.3, zorder=4)
    ax.set_xlabel("Reduced modulus $E_r$ (GPa)")
    ax.set_ylabel("Hardness $H$ (GPa)")
    ax.set_title("Property space (H–E)  ·  ✕ = test mean")
    ax.legend(fontsize=8.5, loc="best", handletextpad=0.3)
    ax.grid(alpha=0.25)
    fig.savefig("fig_property_space.png", bbox_inches="tight")
    plt.show()


# =====================================================================
#  FIGURE 3 — H / E by test: box + raw indents + literature bands
# =====================================================================
def fig_region_comparison(df, expectations=None):
    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2), constrained_layout=True)
    specs = [("Hardness (GPa)", "$H$ (GPa)", True),
             ("Modulus (GPa)",  "$E_r$ (GPa)", False)]
    rng = np.random.default_rng(0)
    for ax, (col, ylabel, show_bands), tag in zip(axes, specs, "ab"):
        data = [df.loc[df["region"] == r, col].dropna().values for r in regions]
        if show_bands and expectations:
            for i, r in enumerate(regions, 1):
                if r in expectations:
                    lo, hi = expectations[r]
                    ax.fill_between([i - 0.42, i + 0.42], lo, hi,
                                    color="0.6", alpha=0.28, zorder=0)
        bp = ax.boxplot(data, positions=range(1, len(regions) + 1), widths=0.55,
                        patch_artist=True, showfliers=False,
                        medianprops=dict(color="k", lw=1.5))
        for patch, r in zip(bp["boxes"], regions):
            patch.set_facecolor(RCOLOR[r]); patch.set_alpha(0.45)
        for i, (r, vals) in enumerate(zip(regions, data), 1):
            ax.scatter(i + rng.uniform(-0.15, 0.15, len(vals)), vals, s=22,
                       color=RCOLOR[r], edgecolor="k", linewidths=0.4, zorder=3)
        ax.set_xticks(range(1, len(regions) + 1))
        ax.set_xticklabels([label_of[r] for r in regions])
        ax.set_ylabel(ylabel)
        ax.set_title(f"({tag}) {col.split()[0]} by test")
        ax.grid(alpha=0.22, axis="y")
    if expectations:
        axes[0].legend(handles=[Patch(facecolor="0.6", alpha=0.28,
                                      label="Literature range")], fontsize=8)
    fig.savefig("fig_region_comparison.png", bbox_inches="tight")
    plt.show()


# ---- run -------------------------------------------------------------
print(f"{len(all_rows)} indents across {all_rows['array'].nunique()} arrays / "
      f"{all_rows['region'].nunique()} tests")

fig_property_map(all_rows)
fig_property_space(all_rows)
fig_region_comparison(all_rows, expectations=LIT_EXPECTATIONS)

# ---- summary table, labeled Test 1..N --------------------------------
summary = (all_rows.groupby("region")
           .agg(n=("Indent", "count"),
                H_mean=("Hardness (GPa)", "mean"), H_std=("Hardness (GPa)", "std"),
                E_mean=("Modulus (GPa)", "mean"), E_std=("Modulus (GPa)", "std"))
           .reindex(regions))
paper_table = summary.assign(**{
    "H (GPa)":   summary.apply(lambda r: f"{r.H_mean:.2f} ± {r.H_std:.2f}", axis=1),
    "E_r (GPa)": summary.apply(lambda r: f"{r.E_mean:.1f} ± {r.E_std:.1f}", axis=1),
})[["n", "H (GPa)", "E_r (GPa)"]]
paper_table.index = [label_of[r] for r in paper_table.index]
display(paper_table)


In [ ]:
all_rows = ui.combined_rows(executions)
print(f"{len(all_rows)} indents across {all_rows['array'].nunique()} arrays / "
      f"{all_rows['region'].nunique()} regions")

ui.plot_property_map(all_rows)
ui.plot_region_comparison(all_rows, expectations=LIT_EXPECTATIONS)

summary = (all_rows.groupby("region")
           .agg(**{"n indents": ("Indent", "count"),
                   "H mean (GPa)": ("Hardness (GPa)", "mean"),
                   "H std": ("Hardness (GPa)", "std"),
                   "E mean (GPa)": ("Modulus (GPa)", "mean"),
                   "E std": ("Modulus (GPa)", "std")})
           .round(2))
display(summary)

In [ ]:
# The orchestrator's final belief about the sample: GP posterior over the full FOV.
final_surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
final_progress = orch.evaluate_progress(state, final_surrogate)

maps = final_surrogate.posterior_maps(n=70)
best = final_surrogate.best_observed()
fig, axes = plt.subplots(2, 2, figsize=(12.5, 9))
panels = [("hardness_mean", "predicted hardness (GPa)", "inferno"),
          ("modulus_mean", "predicted modulus (GPa)", "viridis"),
          ("scalar_mean", "objective score (weighted)", "cividis"),
          ("scalar_sigma", "objective uncertainty (σ)", "magma")]
for ax, (key, title, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(maps[key], origin="lower", extent=maps["extent"], aspect="equal", cmap=cmap)
    ax.scatter(final_surrogate.X[:, 0], final_surrogate.X[:, 1], s=12, c="white",
               edgecolor="k", linewidths=0.4)
    if best:
        ax.scatter([best["x_um"]], [best["y_um"]], marker="*", s=340, c="red",
                   edgecolor="k", linewidths=0.7)
    ax.set_title(title); ax.set_xlabel("stage x (µm)"); ax.set_ylabel("stage y (µm)")
    fig.colorbar(im, ax=ax, shrink=0.8)
fig.suptitle(f"Final GP posterior ({final_surrogate.mode}, {final_surrogate.n_points} indents) — "
             f"★ best site under the objective", y=1.0)
plt.tight_layout(); plt.show()

In [ ]:
# ------------------------- OBJECTIVE ATTAINMENT REPORT -------------------------
ui.agent_banner("Orchestrator Agent", "final objective attainment report")

print("OBJECTIVE:", objective.raw_text, "\n")
display(final_progress.table())

best = final_progress.best_site
if best:
    print(f"\nBest site found: stage ({best['x_um']:.1f}, {best['y_um']:.1f}) µm in '{best['array']}'")
    print(f"  measured H = {best['hardness_gpa']:.2f} GPa, E = {best['modulus_gpa']:.1f} GPa "
          f"(objective score {best['scalar_value']:.2f} vs baseline {final_progress.baseline_scalar:.2f})")
print(f"Improvement over baseline: {final_progress.improvement_pct:.1f}% "
      f"(required: {objective.success.min_improvement_pct:g}%)")
print(f"Budget used: {final_progress.indents_used}/{final_progress.indent_budget} indents in "
      f"{final_progress.round_index} adaptive round(s)")

_decs = state.get("orchestrator_decisions") or []
if _decs:
    print(f"\nFinal orchestrator verdict: {_decs[-1]['action'].upper()} — {_decs[-1]['reason'][:200]}")
    print("\nDecision history:")
    display(pd.DataFrame(_decs)[["round", "action", "proposal_source", "advisor",
                                 "improvement_pct", "indents_used", "reason"]])
else:
    print("\n(No orchestrator rounds were run — use section 14.)")
verdict = "OBJECTIVE MET" if final_progress.all_criteria_met else "OBJECTIVE NOT (YET) MET"
print(f"\n{'='*76}\n{verdict}\n{'='*76}")

## 16. Ground-truth validation (simulator only)

The simulator samples measurements from a hidden material map, so we can check what real hardware
never could: **did the orchestrator's best site actually land on the best phase?**

In [ ]:
if USE_SIMULATOR:
    ui.ground_truth_validation(all_rows)
    if str(REPO_ROOT / "instrument_control") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "instrument_control"))
    from synthetic_microstructure import properties_at
    best = final_surrogate.best_observed()
    if best:
        truth = properties_at(best["x_um"], best["y_um"])
        print(f"Orchestrator's best site ({best['x_um']:.1f}, {best['y_um']:.1f}) µm:")
        print(f"  ground-truth phase:    {truth['phase']}"
              + (" (near boundary)" if truth.get("near_boundary") else ""))
        print(f"  ground-truth H = {truth['hardness_gpa']:.2f} GPa (measured {best['hardness_gpa']:.2f})")
        print(f"  ground-truth E = {truth['modulus_gpa']:.1f} GPa (measured {best['modulus_gpa']:.1f})")
        _want_high = all(g.direction == "maximize" for g in objective.property_goals)
        if _want_high:
            print("  -> objective wanted the HARD phase:",
                  "SUCCESS" if truth["phase"] == "hard_precipitate" or truth.get("near_boundary")
                  else "landed on " + truth["phase"])
else:
    print("Ground-truth validation is only available in simulator mode.")

## 17. Agent conversation — transcript, sequence diagram, audits

Note the new traffic compared to notebook 3: the orchestrator's objective broadcast to all five
roles after Gate 0, its per-round decisions to the operator, and the Acquisition Agent's site
proposals to the Planning Agent.

In [ ]:
msg_log.render_transcript()
msg_log.sequence_diagram()

print("Human approval gates:")
display(ui.ApprovalGate.audit_table())

full_audit = pd.concat(
    [pd.DataFrame(ex["tool_log"]).assign(array=ex["indent_data"]["test_name"]) for ex in executions],
    ignore_index=True)
print(f"Complete instrument tool-call audit ({len(full_audit)} calls, all operator-approved):")
display(full_audit[["array", "tool", "ok", "elapsed_s", "summary"]])

## 18. Run report

The JSON report now includes the compiled objective, the per-role briefings, every orchestrator
decision with provenance, the final progress metrics, and the last acquisition state — a complete
audit of *why* each site was measured, not just *what* was measured.

In [ ]:
from nanoindentation_agents.graph import conclude_node

state.update(conclude_node(state))
print(state["_conclusion_summary"])

report_path = ui.save_run_report(
    REPO_ROOT / "experiment_logs" / f"objective_orchestrator_{time.strftime('%Y%m%d_%H%M%S')}.json",
    operator=OPERATOR_NAME,
    user_objective=USER_OBJECTIVE,
    objective=state["research_objective"],
    orchestrator_briefings=state["orchestrator_briefings"],
    orchestrator_decisions=state.get("orchestrator_decisions", []),
    objective_progress=final_progress.to_dict(),
    acquisition=state.get("acquisition_meta", {}),
    research_question=state["research_question"],
    hypothesis=state["hypothesis"],
    hypothesis_data=state.get("hypothesis_data", {}),
    literature={"query": state.get("literature_search_query"),
                "sources": state.get("literature_sources"),
                "summary": state.get("literature_context")},
    image_analysis={"interpretation": state["image_analysis"].get("interpretation"),
                    "statistics": state["image_analysis"].get("statistical_summary"),
                    "n_targets": len(state.get("measurement_plan", []))},
    indent_arrays=[{k: v for k, v in d.items() if k != "results"} for d in state["indent_history"]],
    measurement_critique=state.get("measurement_critique"),
    final_decision=state.get("measurement_critique_decision"),
    approval_gates=[g.record() for g in ui.ApprovalGate.registry],
    agent_messages=msg_log.entries,
)
print("Report saved:", report_path)

## 19. Shut down the simulator / run a *different* objective

**The point of this architecture:** to run a completely different experiment — e.g.
`"locate the softest, most compliant regions of the coating"` or
`"map areas where hardness is 8-12 GPa"` — change **only `USER_OBJECTIVE`** in section 1 and
re-run. The orchestrator re-compiles the objective, re-briefs every agent, flips the target
ranking and the acquisition direction, and re-derives the critic's success criteria. No other
code changes.

To drive the real iMicro instead: `USE_SIMULATOR = False`, `RUN_SIMULATOR_SERVER = False`,
`.env` server credentials, operator physically present — identical code path.

In [ ]:
if sim_process is not None and sim_process.poll() is None:
    sim_process.terminate()
    try:
        sim_process.wait(timeout=5)
    except Exception:
        sim_process.kill()
    print("Simulator server stopped.")
else:
    print("No notebook-launched simulator process to stop.")

In [ ]:
# ================= PRESENTATION SUMMARY — the objective-driven AI nanoindentation story =================
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

from nanoindentation_agents.critics.measurement_critic import _objective_array_scores

FIG_DIR = REPO_ROOT / "presentation_figures"
FIG_DIR.mkdir(exist_ok=True)

all_rows = ui.combined_rows(executions)
final_surrogate = ObjectiveSurrogate(objective).fit(state["indent_history"])
final_progress = orch.evaluate_progress(state, final_surrogate)
best = final_surrogate.best_observed()
decisions = state.get("orchestrator_decisions") or []
xmin, xmax, ymin, ymax = objective.constraints.fov_bounds

def save(fig, name):
    fig.savefig(FIG_DIR / name, dpi=200, bbox_inches="tight")
    plt.show()

# ---------------- FIG 1: how the architecture works ----------------
fig, ax = plt.subplots(figsize=(14, 5.5)); ax.axis("off")
boxes = [  # (x, y, w, text, color)
    (0.01, 0.62, 0.15, "USER OBJECTIVE\n(plain English goal)", "#f9e79f"),
    (0.20, 0.62, 0.17, "MAIN ORCHESTRATOR\ncompile objective\nbrief all sub-agents", "#a9dfbf"),
    (0.41, 0.80, 0.13, "Image Analysis\nobjective-ranked\ntargets", "#d7bde2"),
    (0.41, 0.55, 0.13, "Literature\npre-seeded query", "#aed6f1"),
    (0.41, 0.30, 0.13, "Hypothesis + Critic\nobjective-directed", "#f5cba7"),
    (0.58, 0.62, 0.14, "HUMAN GATES\n0: objective\n1: hypothesis\n2..N: indents", "#f1948a"),
    (0.76, 0.62, 0.10, "Instrument\nExecutor\n(sim / iMicro)", "#d5dbdb"),
    (0.76, 0.20, 0.10, "Measurement\nCritic\n(objective criteria)", "#d2b4de"),
    (0.20, 0.15, 0.17, "GP SURROGATE + EI\nBayesian optimization\nproposes next sites", "#a3e4d7"),
]
for x, y, w, text, c in boxes:
    ax.add_patch(FancyBboxPatch((x, y), w, 0.20, boxstyle="round,pad=0.012",
                                fc=c, ec="k", lw=1.2, transform=ax.transAxes))
    ax.text(x + w/2, y + 0.10, text, ha="center", va="center", fontsize=9.5,
            weight="bold", transform=ax.transAxes)
arrows = [((0.16, 0.72), (0.20, 0.72)), ((0.37, 0.76), (0.41, 0.87)),
          ((0.37, 0.72), (0.41, 0.64)), ((0.37, 0.67), (0.41, 0.39)),
          ((0.54, 0.87), (0.58, 0.76)), ((0.54, 0.39), (0.58, 0.68)),
          ((0.72, 0.72), (0.76, 0.72)), ((0.81, 0.62), (0.81, 0.40)),
          ((0.76, 0.28), (0.37, 0.24)), ((0.28, 0.35), (0.28, 0.62))]
for a, b in arrows:
    ax.add_patch(FancyArrowPatch(a, b, transform=ax.transAxes, arrowstyle="-|>",
                                 mutation_scale=16, lw=1.4, color="#34495e"))
ax.text(0.52, 0.05, "feedback loop: critic verdict + GP uncertainty + budget → orchestrator decides "
        "continue / refine / objective met / stop", ha="center", fontsize=10, style="italic",
        transform=ax.transAxes)
ax.set_title("Objective-driven autonomous nanoindentation — one goal in, adaptive campaign out",
             fontsize=13, weight="bold")
save(fig, "fig1_architecture.png")

# ---------------- FIG 2: the compiled objective (Gate 0 card) ----------------
fig, ax = plt.subplots(figsize=(11, 4.5)); ax.axis("off")
lines = [f'USER GOAL:  "{objective.raw_text}"', "",
         f"Compiled by: {objective.compiled_by.upper()}    |    Material: {objective.material_system or '-'}", ""]
for g in objective.property_goals:
    lines.append(f"   TARGET:  {g.direction.upper():12s} {g.name:10s} (weight {g.weight:g})"
                 + (f"  range {g.target_range} GPa" if g.target_range else ""))
lines += ["", f"BUDGET:   {objective.constraints.indent_budget} indents | "
          f"{objective.constraints.max_rounds} adaptive rounds | "
          f"{objective.constraints.grid_nx}x{objective.constraints.grid_ny} arrays @ "
          f"{objective.constraints.spacing_um:g} µm",
          f"SUCCESS:  best site ≥ {objective.success.min_improvement_pct:g}% above baseline, "
          f"≥ {objective.success.min_confirming_arrays} arrays, "
          f"≥ {objective.success.min_visited_locations} locations"]
ax.text(0.02, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=11.5,
        transform=ax.transAxes)
ax.set_title("Step 1 — the orchestrator translates the goal into a machine-actionable objective "
             "(human-approved at Gate 0)", fontsize=12, weight="bold", loc="left")
save(fig, "fig2_objective_card.png")

# ---------------- FIG 3: where the AI chose to measure ----------------
img_path = ui.resolve_image_path(state["current_image"])
fig, ax = plt.subplots(figsize=(9.5, 7))
ax.imshow(plt.imread(str(img_path)), extent=(xmin, xmax, ymin, ymax), cmap="gray")
for i, d in enumerate(state["indent_history"]):
    x, y = d["position"]; is_bo = str(d["area_name"]).startswith("bo_")
    ax.scatter(x, y, s=700, facecolor="none", lw=2.5,
               edgecolor="#e74c3c" if is_bo else "#f39c12")
    ax.annotate(f"{i+1}", (x, y), color="white", weight="bold", fontsize=13,
                ha="center", va="center")
    ax.annotate(f"H={d['hardness_mean']:.1f} GPa", (x, y - 6.5), color="white",
                fontsize=9, ha="center")
if best:
    ax.scatter(best["x_um"], best["y_um"], marker="*", s=550, c="lime", edgecolor="k", zorder=5)
ax.legend(handles=[
    mpatches.Patch(color="#f39c12", label="round 1 — image-analysis targets (objective-ranked)"),
    mpatches.Patch(color="#e74c3c", label="adaptive rounds — GP / Expected Improvement"),
    mpatches.Patch(color="lime", label="best site under the objective")], loc="lower right", fontsize=9)
ax.set_xlabel("stage x (µm)"); ax.set_ylabel("stage y (µm)")
ax.set_title("Step 2 — measurement sites in execution order: AI vision seeds round 1, "
             "Bayesian optimization takes over", fontsize=12, weight="bold")
save(fig, "fig3_sites_on_sample.png")

# ---------------- FIG 4: campaign timeline ----------------
hist = state["indent_history"]; names = [d["area_name"] for d in hist]; idx = np.arange(len(hist))
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
for ax, key, skey, unit, color in [(axes[0], "hardness_mean", "hardness_std", "H (GPa)", "#c0392b"),
                                   (axes[1], "modulus_mean", "modulus_std", "E (GPa)", "#2471a3")]:
    v = [d[key] for d in hist]; s = [d.get(skey, 0) for d in hist]
    ax.errorbar(idx, v, yerr=s, fmt="o-", ms=9, capsize=4, lw=1.6, color=color)
    ax.axhline(np.mean(v), ls="--", color="gray", label=f"sample mean = {np.mean(v):.1f}")
    ax.set_ylabel(unit); ax.legend(fontsize=9); ax.grid(alpha=0.3)
axes[1].set_xticks(idx); axes[1].set_xticklabels(names, rotation=20, ha="right", fontsize=9)
axes[0].set_title("Step 3 — properties measured array by array (error bars = per-array spread)",
                  fontsize=12, weight="bold")
save(fig, "fig4_campaign_timeline.png")

# ---------------- FIG 5: objective score per array vs acceptance threshold ----------------
scores = np.array(_objective_array_scores(hist, orch.measurement_criteria())) * 100
thr = objective.success.min_improvement_pct
fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ["#27ae60" if s == scores.max() else "#95a5a6" for s in scores]
ax.bar(idx, scores, color=colors, edgecolor="k")
ax.axhline(thr, color="#c0392b", lw=2, ls="--", label=f"acceptance threshold (+{thr:g}%)")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(idx); ax.set_xticklabels(names, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("weighted objective improvement\nvs sample baseline (%)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("Step 4 — how each array scores against the objective "
             "(green = winner; H and E combined by the objective's weights)",
             fontsize=12, weight="bold")
save(fig, "fig5_objective_scores.png")

# ---------------- FIG 6: what the AI believes — GP posterior + acquisition ----------------
maps = final_surrogate.posterior_maps(n=70)
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
panels = [("hardness_mean", "predicted hardness (GPa)", "inferno"),
          ("modulus_mean", "predicted modulus (GPa)", "viridis"),
          ("scalar_mean", "objective score (weighted)", "cividis"),
          ("scalar_sigma", "uncertainty σ (unexplored = bright)", "magma"),
          ("ei", "Expected Improvement (where to go next)", "plasma")]
for ax, (key, title, cmap) in zip(axes.ravel(), panels):
    im = ax.imshow(maps[key], origin="lower", extent=maps["extent"], aspect="equal", cmap=cmap)
    ax.scatter(final_surrogate.X[:, 0], final_surrogate.X[:, 1], s=10, c="white",
               edgecolor="k", linewidths=0.3)
    if best:
        ax.scatter(best["x_um"], best["y_um"], marker="*", s=300, c="red", edgecolor="k")
    ax.set_title(title, fontsize=10.5); fig.colorbar(im, ax=ax, shrink=0.8)
axes[1, 2].axis("off")
axes[1, 2].text(0.05, 0.75, f"Gaussian-process surrogate\n({final_surrogate.mode})\n\n"
                f"{final_surrogate.n_points} indents observed\n"
                f"white dots = measurements\n★ = best site found\n\n"
                "The AI 'imagines' the whole\nsample from few measurements,\n"
                "then indents where Expected\nImprovement is highest.",
                fontsize=11, va="top", transform=axes[1, 2].transAxes)
fig.suptitle("Step 5 — the AI's mental model of the sample and where it decided to look next",
             fontsize=13, weight="bold")
plt.tight_layout(); save(fig, "fig6_gp_posterior.png")

# ---------------- FIG 7: the autonomy dashboard — decisions, budget, convergence ----------------
if decisions:
    rounds = [d["round"] + 1 for d in decisions]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].step(rounds, [d["indents_used"] for d in decisions], "o-", where="post", lw=2)
    axes[0].axhline(objective.constraints.indent_budget, color="#c0392b", ls="--",
                    label=f"budget = {objective.constraints.indent_budget}")
    axes[0].set_xlabel("orchestrator round"); axes[0].set_ylabel("indents used")
    axes[0].set_title("resource use"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(rounds, [d["improvement_pct"] for d in decisions], "o-", lw=2, color="#27ae60")
    axes[1].axhline(thr, color="#c0392b", ls="--", label=f"required +{thr:g}%")
    axes[1].set_xlabel("orchestrator round"); axes[1].set_ylabel("best improvement (%)")
    axes[1].set_title("progress toward the objective"); axes[1].legend(); axes[1].grid(alpha=0.3)
    for ax in axes:
        ax.annotate(f"FINAL: {decisions[-1]['action'].upper()}",
                    xy=(0.98, 0.06), xycoords="axes fraction", ha="right", fontsize=11,
                    weight="bold", color="#1a5276")
    fig.suptitle("Step 6 — the orchestrator's autonomous decisions "
                 f"({' → '.join(d['action'] for d in decisions)})", fontsize=12, weight="bold")
    plt.tight_layout(); save(fig, "fig7_autonomy_dashboard.png")

# ---------------- FIG 8: final scorecard ----------------
fig, ax = plt.subplots(figsize=(10, 4)); ax.axis("off")
y = 0.88
for name, st in final_progress.criteria_status.items():
    ok = st["met"]
    ax.add_patch(FancyBboxPatch((0.02, y - 0.07), 0.10, 0.12, boxstyle="round,pad=0.01",
                                fc="#27ae60" if ok else "#c0392b", transform=ax.transAxes))
    ax.text(0.07, y - 0.01, "MET" if ok else "NOT MET", color="white", ha="center",
            weight="bold", fontsize=10, transform=ax.transAxes)
    ax.text(0.15, y - 0.01, f"{name}:  {st['detail']}", fontsize=11.5, va="center",
            transform=ax.transAxes)
    y -= 0.20
verdict = "OBJECTIVE MET" if final_progress.all_criteria_met else "OBJECTIVE NOT (YET) MET"
extra = (f"   |   best site ({best['x_um']:.0f}, {best['y_um']:.0f}) µm: "
         f"H = {best['hardness_gpa']:.1f} GPa, E = {best['modulus_gpa']:.0f} GPa" if best else "")
ax.text(0.02, y - 0.02, f"{verdict}   |   {final_progress.indents_used}/"
        f"{final_progress.indent_budget} indents used{extra}",
        fontsize=13, weight="bold", color="#1a5276", transform=ax.transAxes)
ax.set_title("Step 7 — final scorecard against the user's acceptance criteria",
             fontsize=12, weight="bold", loc="left")
save(fig, "fig8_scorecard.png")

# ---------------- extras rendered by existing helpers ----------------
if USE_SIMULATOR:
    ui.ground_truth_validation(all_rows)      # proof the AI targeted real microstructure
msg_log.sequence_diagram()                     # who talked to whom, in order

print(f"\nAll figures saved to: {FIG_DIR}")
for p in sorted(FIG_DIR.glob('fig*.png')):
    print("  -", p.name)


## FINAL ANALYSIS AGENT

In [ ]:
# ================= FINAL ANALYSIS AGENT — expert review of all completed tests =================
from nanoindentation_agents.agents.final_analysis_agent import FinalAnalysisAgent

ui.agent_banner("Final Analysis Agent", "reviewing every completed test like a nanoindentation expert")

all_rows = ui.combined_rows(executions)

analyst = FinalAnalysisAgent(objective, cfg, use_llm=TRY_LLM)
report = analyst.analyze(
    all_rows,
    state=state,
    expectations=LIT_EXPECTATIONS,
    progress=final_progress,          # from the objective-attainment cell above
)

report.display()

# record the expert read-out into the agent-to-agent message log + run state
msg_log.log("Final Analysis Agent", "Orchestrator Agent",
            "Campaign review complete — findings, conclusion, limitations, next steps.",
            detail=report.conclusion[:280])
state["final_analysis"] = report.to_dict()
